In [1]:
# ============================================================
# CELL 1 — ENVIRONMENT, IMPORTS, SEED AND GPU CHECK
# DenseNet121 Mouth ROI Deepfake Classification
# ============================================================

import os
import sys
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ------------------------------------------------------------
# Version information
# ------------------------------------------------------------
print("=" * 65)
print("DENSENET121 MOUTH ROI DEEPFAKE EXPERIMENT")
print("=" * 65)

print("Python version     :", sys.version.split()[0])
print("TensorFlow version :", tf.__version__)
print("NumPy version      :", np.__version__)
print("Random seed        :", SEED)

# ------------------------------------------------------------
# GPU check
# ------------------------------------------------------------
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("\nGPU detected:")

    for gpu in gpus:
        print(" -", gpu)

    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        print("GPU memory growth  : ENABLED")

    except RuntimeError as error:
        print("GPU configuration warning:", error)

else:
    print("\nWARNING: GPU could not be detected.")
    print("Colab menu:")
    print("Runtime > Change runtime type > T4 GPU")

# ------------------------------------------------------------
# Experiment device
# ------------------------------------------------------------
device_name = tf.test.gpu_device_name()

print("\nTensorFlow device  :", device_name if device_name else "CPU")
print("=" * 65)
print("CELL 1 COMPLETED")

DENSENET121 MOUTH ROI DEEPFAKE EXPERIMENT
Python version     : 3.12.13
TensorFlow version : 2.20.0
NumPy version      : 2.0.2
Random seed        : 42

GPU detected:
 - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
GPU memory growth  : ENABLED

TensorFlow device  : /device:GPU:0
CELL 1 COMPLETED


In [4]:
# ============================================================
# CELL 2 — GOOGLE DRIVE AND EXPERIMENT FOLDER DETECTION
# ============================================================

from google.colab import drive
from pathlib import Path
import unicodedata

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")


MY_DRIVE = Path("/content/drive/MyDrive")


# ------------------------------------------------------------
# Unicode-safe folder comparison
# ------------------------------------------------------------
def normalize_name(name):
    """
    Türkçe karakterlerin farklı Unicode gösterimlerinden
    kaynaklanan klasör yolu sorunlarını önler.
    """
    return unicodedata.normalize(
        "NFC",
        str(name)
    ).strip().casefold()


def find_child_folder(parent, expected_name):
    """
    Verilen klasörün altında hedef klasörü Unicode uyumlu
    biçimde bulur.
    """
    if not parent.exists():
        raise FileNotFoundError(
            f"Parent folder not found: {parent}"
        )

    expected_normalized = normalize_name(expected_name)

    folders = [
        item for item in parent.iterdir()
        if item.is_dir()
    ]

    for folder in folders:
        if normalize_name(folder.name) == expected_normalized:
            return folder

    available_names = [folder.name for folder in folders]

    raise FileNotFoundError(
        f"\nFolder not found: {expected_name}\n"
        f"Parent folder: {parent}\n"
        f"Available folders: {available_names}"
    )


# ------------------------------------------------------------
# Find the real experiment folders
# ------------------------------------------------------------

# MyDrive/AISC DeepFake Çalışmaları
AISC_ROOT = find_child_folder(
    MY_DRIVE,
    "AISC DeepFake Çalışmaları"
)

# MyDrive/AISC DeepFake Çalışmaları/Deneyler
EXPERIMENTS_ROOT = find_child_folder(
    AISC_ROOT,
    "Deneyler"
)

# MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara
DILARA_ROOT = find_child_folder(
    EXPERIMENTS_ROOT,
    "Dilara"
)

# MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1
EXPERIMENT_ROOT = find_child_folder(
    DILARA_ROOT,
    "Deney 1"
)


# ------------------------------------------------------------
# Find or create Results folder
# ------------------------------------------------------------
try:
    RESULTS_ROOT = find_child_folder(
        EXPERIMENT_ROOT,
        "Sonuçlar"
    )

except FileNotFoundError:
    RESULTS_ROOT = EXPERIMENT_ROOT / "Sonuçlar"
    RESULTS_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Print detected paths
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("GOOGLE DRIVE AND EXPERIMENT FOLDER CHECK")
print("=" * 70)

print("MyDrive root     :", MY_DRIVE)
print("AISC root        :", AISC_ROOT)
print("Experiments root :", EXPERIMENTS_ROOT)
print("Dilara root      :", DILARA_ROOT)
print("Experiment root  :", EXPERIMENT_ROOT)
print("Results root     :", RESULTS_ROOT)

print("\nExperiment folder: FOUND")
print("Results folder   : READY")


# ------------------------------------------------------------
# Show all items inside Experiment 1
# ------------------------------------------------------------
print("\nFiles and folders inside Experiment 1:")
print("-" * 70)

items = sorted(
    EXPERIMENT_ROOT.iterdir(),
    key=lambda item: normalize_name(item.name)
)

if len(items) == 0:
    print("The Experiment 1 folder is empty.")

else:
    for item in items:

        if item.is_dir():
            item_type = "FOLDER"
        else:
            item_type = "FILE"

        print(f"[{item_type:6}] {item.name}")


print("=" * 70)
print("CELL 2 COMPLETED")

Google Drive is already mounted.

GOOGLE DRIVE AND EXPERIMENT FOLDER CHECK
MyDrive root     : /content/drive/MyDrive
AISC root        : /content/drive/MyDrive/AISC DeepFake Çalışmaları
Experiments root : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler
Dilara root      : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara
Experiment root  : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1
Results root     : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar

Experiment folder: FOUND
Results folder   : READY

Files and folders inside Experiment 1:
----------------------------------------------------------------------
[FOLDER] Ağız
[FOLDER] Kod Açıklamaları
[FOLDER] Kodlar
[FOLDER] Raporlar
[FOLDER] Sonuçlar
CELL 2 COMPLETED


In [5]:
# ============================================================
# CELL 3 — MOUTH DATASET FOLDER INSPECTION
# ============================================================

from pathlib import Path
from collections import Counter

# ------------------------------------------------------------
# Find the Mouth folder
# ------------------------------------------------------------
MOUTH_ROOT = find_child_folder(
    EXPERIMENT_ROOT,
    "Ağız"
)

# Supported image extensions
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

print("\n" + "=" * 75)
print("MOUTH DATASET FOLDER INSPECTION")
print("=" * 75)

print("Mouth root:", MOUTH_ROOT)

# ------------------------------------------------------------
# Show folder tree
# Maximum depth: 4
# ------------------------------------------------------------
print("\nFolder structure:")
print("-" * 75)

all_folders = sorted(
    [
        path for path in MOUTH_ROOT.rglob("*")
        if path.is_dir()
    ],
    key=lambda path: normalize_name(str(path))
)

if not all_folders:
    print("No subfolders found inside the Mouth folder.")

else:
    for folder in all_folders:

        relative_path = folder.relative_to(MOUTH_ROOT)
        depth = len(relative_path.parts)

        if depth <= 4:

            direct_image_count = sum(
                1
                for file in folder.iterdir()
                if (
                    file.is_file()
                    and file.suffix.lower() in IMAGE_EXTENSIONS
                )
            )

            indentation = "    " * (depth - 1)

            print(
                f"{indentation}[FOLDER] {relative_path} "
                f"| Direct images: {direct_image_count}"
            )

# ------------------------------------------------------------
# Count all images recursively
# ------------------------------------------------------------
all_images = [
    path
    for path in MOUTH_ROOT.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )
]

extension_counts = Counter(
    path.suffix.lower()
    for path in all_images
)

print("\n" + "-" * 75)
print("IMAGE SUMMARY")
print("-" * 75)

print("Total mouth images:", len(all_images))
print("Image extensions  :", dict(extension_counts))

# ------------------------------------------------------------
# Show sample image paths
# ------------------------------------------------------------
print("\nSample image paths:")
print("-" * 75)

for sample_path in all_images[:10]:
    print(sample_path.relative_to(MOUTH_ROOT))

print("=" * 75)
print("CELL 3 COMPLETED")


MOUTH DATASET FOLDER INSPECTION
Mouth root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Ağız

Folder structure:
---------------------------------------------------------------------------
[FOLDER] HOG_LBP_KAZE_Results | Direct images: 0
    [FOLDER] HOG_LBP_KAZE_Results/feature_chunks | Direct images: 0
    [FOLDER] HOG_LBP_KAZE_Results/figures | Direct images: 3
[FOLDER] mouth_roi_output | Direct images: 0
    [FOLDER] mouth_roi_output/fake | Direct images: 0
        [FOLDER] mouth_roi_output/fake/test | Direct images: 0
            [FOLDER] mouth_roi_output/fake/test/debug | Direct images: 156
            [FOLDER] mouth_roi_output/fake/test/landmarks | Direct images: 0
            [FOLDER] mouth_roi_output/fake/test/mouth | Direct images: 156
        [FOLDER] mouth_roi_output/fake/train | Direct images: 0
            [FOLDER] mouth_roi_output/fake/train/debug | Direct images: 1192
            [FOLDER] mouth_roi_output/fake/train/landmarks | Direct ima

In [6]:
# ============================================================
# CELL 4 — DATASET PATHS, LABELS AND RUN DIRECTORIES
# ============================================================

from pathlib import Path
from datetime import datetime
import json

# ------------------------------------------------------------
# Dataset root
# ------------------------------------------------------------
DATASET_ROOT = MOUTH_ROOT / "mouth_roi_output"

# Class labels
CLASS_NAMES = ["real", "fake"]

# Label mapping
# real = 0
# fake = 1
LABEL_MAP = {
    "real": 0,
    "fake": 1
}

# Supported image formats
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


# ------------------------------------------------------------
# Exact mouth ROI folders
# ------------------------------------------------------------
DATA_FOLDERS = {
    "train": {
        "real": DATASET_ROOT / "real" / "train" / "mouth",
        "fake": DATASET_ROOT / "fake" / "train" / "mouth",
    },

    "val": {
        "real": DATASET_ROOT / "real" / "val" / "mouth",
        "fake": DATASET_ROOT / "fake" / "val" / "mouth",
    },

    "test": {
        "real": DATASET_ROOT / "real" / "test" / "mouth",
        "fake": DATASET_ROOT / "fake" / "test" / "mouth",
    }
}


# ------------------------------------------------------------
# Helper function: collect image files
# ------------------------------------------------------------
def collect_image_files(folder):
    if not folder.exists():
        raise FileNotFoundError(
            f"Dataset folder not found:\n{folder}"
        )

    image_files = sorted(
        [
            file
            for file in folder.iterdir()
            if (
                file.is_file()
                and file.suffix.lower() in IMAGE_EXTENSIONS
            )
        ],
        key=lambda file: file.name.lower()
    )

    return image_files


# ------------------------------------------------------------
# Collect paths and labels
# ------------------------------------------------------------
dataset_records = {}

for split_name, class_folders in DATA_FOLDERS.items():

    split_paths = []
    split_labels = []

    for class_name in CLASS_NAMES:

        class_folder = class_folders[class_name]
        class_files = collect_image_files(class_folder)
        class_label = LABEL_MAP[class_name]

        split_paths.extend(
            [str(file) for file in class_files]
        )

        split_labels.extend(
            [class_label] * len(class_files)
        )

    dataset_records[split_name] = {
        "paths": split_paths,
        "labels": split_labels
    }


# ------------------------------------------------------------
# Count verification
# ------------------------------------------------------------
EXPECTED_COUNTS = {
    "train": {
        "real": 1197,
        "fake": 1192,
        "total": 2389
    },

    "val": {
        "real": 155,
        "fake": 141,
        "total": 296
    },

    "test": {
        "real": 146,
        "fake": 156,
        "total": 302
    }
}


print("\n" + "=" * 75)
print("DENSENET121 DATASET PATH AND COUNT CHECK")
print("=" * 75)

actual_counts = {}

for split_name in ["train", "val", "test"]:

    labels = dataset_records[split_name]["labels"]

    real_count = labels.count(LABEL_MAP["real"])
    fake_count = labels.count(LABEL_MAP["fake"])
    total_count = len(labels)

    actual_counts[split_name] = {
        "real": real_count,
        "fake": fake_count,
        "total": total_count
    }

    print(
        f"{split_name.upper():5} | "
        f"Real: {real_count:4} | "
        f"Fake: {fake_count:4} | "
        f"Total: {total_count:4}"
    )

    assert actual_counts[split_name] == EXPECTED_COUNTS[split_name], (
        f"{split_name} count mismatch.\n"
        f"Expected: {EXPECTED_COUNTS[split_name]}\n"
        f"Actual  : {actual_counts[split_name]}"
    )


# ------------------------------------------------------------
# Duplicate path check
# ------------------------------------------------------------
train_path_set = set(dataset_records["train"]["paths"])
val_path_set = set(dataset_records["val"]["paths"])
test_path_set = set(dataset_records["test"]["paths"])

assert train_path_set.isdisjoint(val_path_set), (
    "The same image path exists in train and validation."
)

assert train_path_set.isdisjoint(test_path_set), (
    "The same image path exists in train and test."
)

assert val_path_set.isdisjoint(test_path_set), (
    "The same image path exists in validation and test."
)


# ------------------------------------------------------------
# Create a unique experiment run folder
# ------------------------------------------------------------
RUN_ID = datetime.now().strftime(
    "%Y%m%d_%H%M%S_densenet121_mouth_seed42"
)

MODEL_RESULTS_ROOT = (
    RESULTS_ROOT /
    "DenseNet121_Mouth_Results"
)

RUN_ROOT = MODEL_RESULTS_ROOT / RUN_ID

MODEL_DIR = RUN_ROOT / "models"
FIGURE_DIR = RUN_ROOT / "figures"
TABLE_DIR = RUN_ROOT / "tables"
LOG_DIR = RUN_ROOT / "logs"
PREDICTION_DIR = RUN_ROOT / "predictions"

for folder in [
    MODEL_DIR,
    FIGURE_DIR,
    TABLE_DIR,
    LOG_DIR,
    PREDICTION_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Save initial experiment configuration
# ------------------------------------------------------------
initial_config = {
    "run_id": RUN_ID,
    "model_name": "DenseNet121",
    "task": "Mouth ROI Deepfake Binary Classification",
    "seed": SEED,
    "label_mapping": LABEL_MAP,
    "dataset_root": str(DATASET_ROOT),
    "dataset_counts": actual_counts,
    "results_root": str(RUN_ROOT)
}

CONFIG_PATH = RUN_ROOT / "initial_config.json"

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as config_file:
    json.dump(
        initial_config,
        config_file,
        indent=4,
        ensure_ascii=False
    )


print("-" * 75)
print("All dataset counts       : VERIFIED")
print("Duplicate path check     : PASS")
print("Debug images             : EXCLUDED")
print("Landmark files           : EXCLUDED")
print("Run ID                   :", RUN_ID)
print("Run results folder       :", RUN_ROOT)
print("Initial configuration    :", CONFIG_PATH)
print("=" * 75)
print("CELL 4 COMPLETED")


DENSENET121 DATASET PATH AND COUNT CHECK
TRAIN | Real: 1197 | Fake: 1192 | Total: 2389
VAL   | Real:  155 | Fake:  141 | Total:  296
TEST  | Real:  146 | Fake:  156 | Total:  302
---------------------------------------------------------------------------
All dataset counts       : VERIFIED
Duplicate path check     : PASS
Debug images             : EXCLUDED
Landmark files           : EXCLUDED
Run ID                   : 20260808_085615_densenet121_mouth_seed42
Run results folder       : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_085615_densenet121_mouth_seed42
Initial configuration    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_085615_densenet121_mouth_seed42/initial_config.json
CELL 4 COMPLETED


In [7]:
# ============================================================
# CELL 5 — TF.DATA PIPELINE AND DATA AUGMENTATION
# ============================================================

import tensorflow as tf
from tensorflow.keras.applications.densenet import preprocess_input

# ------------------------------------------------------------
# Image and batch configuration
# ------------------------------------------------------------
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_SIZE = (IMAGE_HEIGHT, IMAGE_WIDTH)

BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

print("\n" + "=" * 75)
print("TF.DATA PIPELINE AND DENSENET121 PREPROCESSING")
print("=" * 75)

print("Image size :", IMAGE_SIZE)
print("Batch size :", BATCH_SIZE)


# ------------------------------------------------------------
# Controlled data augmentation
# Only for the training dataset
# ------------------------------------------------------------
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip(
            mode="horizontal",
            seed=SEED
        ),

        tf.keras.layers.RandomRotation(
            factor=0.03,
            fill_mode="reflect",
            seed=SEED
        ),

        tf.keras.layers.RandomZoom(
            height_factor=(-0.08, 0.08),
            width_factor=(-0.08, 0.08),
            fill_mode="reflect",
            seed=SEED
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="reflect",
            seed=SEED
        ),

        tf.keras.layers.RandomContrast(
            factor=0.10,
            seed=SEED
        ),
    ],
    name="mouth_data_augmentation"
)


# ------------------------------------------------------------
# Image decoding function
# ------------------------------------------------------------
def decode_and_resize_image(image_path, label):
    """
    Reads an image, converts it to RGB and resizes it to
    the DenseNet121 input resolution.
    """

    image_bytes = tf.io.read_file(image_path)

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        size=IMAGE_SIZE,
        method=tf.image.ResizeMethod.BILINEAR,
        antialias=True
    )

    image = tf.cast(
        image,
        tf.float32
    )

    label = tf.cast(
        label,
        tf.float32
    )

    return image, label


# ------------------------------------------------------------
# Training preprocessing
# ------------------------------------------------------------
def preprocess_training_image(image, label):
    """
    Applies augmentation and DenseNet121 preprocessing.
    """

    image = data_augmentation(
        image,
        training=True
    )

    image = tf.clip_by_value(
        image,
        0.0,
        255.0
    )

    image = preprocess_input(image)

    return image, label


# ------------------------------------------------------------
# Validation and test preprocessing
# ------------------------------------------------------------
def preprocess_evaluation_image(image, label):
    """
    Applies only DenseNet121 preprocessing.
    No augmentation is used.
    """

    image = preprocess_input(image)

    return image, label


# ------------------------------------------------------------
# Dataset creation function
# ------------------------------------------------------------
def create_tf_dataset(
    image_paths,
    labels,
    training=False
):
    """
    Creates a TensorFlow dataset from image paths and labels.
    """

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            image_paths,
            labels
        )
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(image_paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        decode_and_resize_image,
        num_parallel_calls=AUTOTUNE,
        deterministic=not training
    )

    if training:
        dataset = dataset.map(
            preprocess_training_image,
            num_parallel_calls=AUTOTUNE,
            deterministic=False
        )

    else:
        dataset = dataset.map(
            preprocess_evaluation_image,
            num_parallel_calls=AUTOTUNE,
            deterministic=True
        )

    dataset = dataset.batch(
        BATCH_SIZE,
        drop_remainder=False
    )

    dataset = dataset.prefetch(
        AUTOTUNE
    )

    return dataset


# ------------------------------------------------------------
# Create train, validation and test datasets
# ------------------------------------------------------------
train_dataset = create_tf_dataset(
    dataset_records["train"]["paths"],
    dataset_records["train"]["labels"],
    training=True
)

val_dataset = create_tf_dataset(
    dataset_records["val"]["paths"],
    dataset_records["val"]["labels"],
    training=False
)

test_dataset = create_tf_dataset(
    dataset_records["test"]["paths"],
    dataset_records["test"]["labels"],
    training=False
)


# ------------------------------------------------------------
# Dataset cardinalities
# ------------------------------------------------------------
train_batches = int(
    tf.data.experimental.cardinality(
        train_dataset
    ).numpy()
)

val_batches = int(
    tf.data.experimental.cardinality(
        val_dataset
    ).numpy()
)

test_batches = int(
    tf.data.experimental.cardinality(
        test_dataset
    ).numpy()
)


# ------------------------------------------------------------
# Read one validation batch for smoke testing
# ------------------------------------------------------------
sample_images, sample_labels = next(
    iter(val_dataset)
)

print("\nDataset batches:")
print("Train batches      :", train_batches)
print("Validation batches :", val_batches)
print("Test batches       :", test_batches)

print("\nSmoke-test batch:")
print("Image batch shape  :", sample_images.shape)
print("Label batch shape  :", sample_labels.shape)
print("Image dtype        :", sample_images.dtype)
print("Label dtype        :", sample_labels.dtype)

print(
    "Preprocessed range :",
    float(tf.reduce_min(sample_images)),
    "to",
    float(tf.reduce_max(sample_images))
)

print(
    "Sample labels      :",
    sample_labels.numpy().astype(int).tolist()
)


# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------
assert sample_images.shape[1:] == (
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    3
), "Image shape is incorrect."

assert sample_labels.shape[0] <= BATCH_SIZE, (
    "Batch size is incorrect."
)

assert train_batches == 150, (
    f"Unexpected number of train batches: {train_batches}"
)

assert val_batches == 19, (
    f"Unexpected number of validation batches: {val_batches}"
)

assert test_batches == 19, (
    f"Unexpected number of test batches: {test_batches}"
)


print("-" * 75)
print("Training augmentation : ENABLED")
print("Validation augmentation: DISABLED")
print("Test augmentation      : DISABLED")
print("DenseNet preprocessing : ENABLED")
print("Dataset smoke test     : PASS")
print("=" * 75)
print("CELL 5 COMPLETED")


TF.DATA PIPELINE AND DENSENET121 PREPROCESSING
Image size : (224, 224)
Batch size : 16

Dataset batches:
Train batches      : 150
Validation batches : 19
Test batches       : 19

Smoke-test batch:
Image batch shape  : (16, 224, 224, 3)
Label batch shape  : (16,)
Image dtype        : <dtype: 'float32'>
Label dtype        : <dtype: 'float32'>
Preprocessed range : -2.1179039478302 to 2.640000104904175
Sample labels      : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
---------------------------------------------------------------------------
Training augmentation : ENABLED
Validation augmentation: DISABLED
Test augmentation      : DISABLED
DenseNet preprocessing : ENABLED
Dataset smoke test     : PASS
CELL 5 COMPLETED


In [8]:
# ============================================================
# CELL 6 — SOURCE FILENAME AND ID FORMAT AUDIT
# ============================================================

from pathlib import Path
from collections import Counter
import re

print("\n" + "=" * 80)
print("SOURCE FILENAME AND ID FORMAT AUDIT")
print("=" * 80)


# ------------------------------------------------------------
# Show representative filenames from every split and class
# ------------------------------------------------------------
for split_name in ["train", "val", "test"]:

    print(f"\n[{split_name.upper()}]")

    split_paths = dataset_records[split_name]["paths"]
    split_labels = dataset_records[split_name]["labels"]

    for class_name, class_label in LABEL_MAP.items():

        class_paths = [
            Path(path)
            for path, label in zip(
                split_paths,
                split_labels
            )
            if label == class_label
        ]

        print(
            f"\nClass: {class_name} "
            f"| Total: {len(class_paths)}"
        )

        # First five samples
        for sample_path in class_paths[:5]:
            print("  FIRST :", sample_path.name)

        # Last three samples
        for sample_path in class_paths[-3:]:
            print("  LAST  :", sample_path.name)


# ------------------------------------------------------------
# Filename token analysis
# ------------------------------------------------------------
all_dataset_paths = []

for split_name in ["train", "val", "test"]:
    all_dataset_paths.extend(
        [
            Path(path)
            for path in dataset_records[split_name]["paths"]
        ]
    )

extension_counts = Counter(
    path.suffix.lower()
    for path in all_dataset_paths
)

underscore_token_counts = Counter(
    len(path.stem.split("_"))
    for path in all_dataset_paths
)

print("\n" + "-" * 80)
print("GENERAL FILENAME SUMMARY")
print("-" * 80)

print("Total files             :", len(all_dataset_paths))
print("Extension distribution  :", dict(extension_counts))
print(
    "Underscore token counts :",
    dict(
        sorted(
            underscore_token_counts.items()
        )
    )
)


# ------------------------------------------------------------
# Search for common frame indicators
# ------------------------------------------------------------
frame_patterns = {
    "contains_frame": re.compile(
        r"frame",
        flags=re.IGNORECASE
    ),

    "ends_with_number": re.compile(
        r"\d+$"
    ),

    "contains_multiple_numbers": re.compile(
        r".*\d+.*\d+.*"
    )
}

pattern_results = {}

for pattern_name, pattern in frame_patterns.items():

    matched_count = sum(
        1
        for path in all_dataset_paths
        if pattern.search(path.stem)
    )

    pattern_results[pattern_name] = matched_count

print("\nFilename pattern counts:")

for pattern_name, matched_count in pattern_results.items():
    print(
        f"{pattern_name:28}: "
        f"{matched_count} / {len(all_dataset_paths)}"
    )


# ------------------------------------------------------------
# Check filename uniqueness
# ------------------------------------------------------------
all_filenames = [
    path.name
    for path in all_dataset_paths
]

unique_filename_count = len(
    set(all_filenames)
)

duplicate_filename_count = (
    len(all_filenames) -
    unique_filename_count
)

print("\nFilename uniqueness:")
print("Total filenames         :", len(all_filenames))
print("Unique filenames        :", unique_filename_count)
print("Repeated filename count :", duplicate_filename_count)


print("\n" + "=" * 80)
print("CELL 6 COMPLETED")
print("=" * 80)


SOURCE FILENAME AND ID FORMAT AUDIT

[TRAIN]

Class: real | Total: 1197
  FIRST : real_train_real_train_00000_face00.png
  FIRST : real_train_real_train_00001_face00.png
  FIRST : real_train_real_train_00002_face00.png
  FIRST : real_train_real_train_00004_face00.png
  FIRST : real_train_real_train_00005_face00.png
  LAST  : real_train_real_train_01197_face00.png
  LAST  : real_train_real_train_01198_face00.png
  LAST  : real_train_real_train_01199_face00.png

Class: fake | Total: 1192
  FIRST : fake_train_01130_face00.png
  FIRST : fake_train_fake_train_00000_face00.png
  FIRST : fake_train_fake_train_00001_face00.png
  FIRST : fake_train_fake_train_00002_face00.png
  FIRST : fake_train_fake_train_00003_face00.png
  LAST  : fake_train_fake_train_01197_face00.png
  LAST  : fake_train_fake_train_01198_face00.png
  LAST  : fake_train_fake_train_01199_face00.png

[VAL]

Class: real | Total: 155
  FIRST : real_val_real_val_00000_face00.png
  FIRST : real_val_real_val_00001_face00.png
  FI

In [9]:
# ============================================================
# CELL 7 — METADATA AND SPLIT MANIFEST SEARCH
# ============================================================

from pathlib import Path
from collections import Counter

print("\n" + "=" * 80)
print("METADATA AND SPLIT MANIFEST SEARCH")
print("=" * 80)

# ------------------------------------------------------------
# Metadata-like file extensions
# ------------------------------------------------------------
METADATA_EXTENSIONS = {
    ".csv",
    ".json",
    ".jsonl",
    ".parquet",
    ".yaml",
    ".yml",
    ".txt",
    ".pkl",
    ".pickle",
    ".npy",
    ".npz"
}

# Search inside the complete mouth experiment folder
metadata_candidates = sorted(
    [
        file
        for file in MOUTH_ROOT.rglob("*")
        if (
            file.is_file()
            and file.suffix.lower() in METADATA_EXTENSIONS
        )
    ],
    key=lambda file: normalize_name(str(file))
)

print("Search root:", MOUTH_ROOT)
print(
    "Metadata-like files found:",
    len(metadata_candidates)
)

# ------------------------------------------------------------
# Print candidate metadata files
# ------------------------------------------------------------
if metadata_candidates:

    print("\nCandidate metadata files:")
    print("-" * 80)

    for index, file in enumerate(
        metadata_candidates,
        start=1
    ):
        relative_path = file.relative_to(
            MOUTH_ROOT
        )

        file_size_kb = file.stat().st_size / 1024

        print(
            f"{index:03d} | "
            f"{relative_path} | "
            f"{file_size_kb:.2f} KB"
        )

else:
    print("\nNo metadata-like files were found.")


# ------------------------------------------------------------
# Search filenames containing important keywords
# ------------------------------------------------------------
IMPORTANT_KEYWORDS = {
    "metadata",
    "manifest",
    "split",
    "source",
    "video",
    "frame",
    "index",
    "summary",
    "audit"
}

keyword_candidates = sorted(
    [
        file
        for file in MOUTH_ROOT.rglob("*")
        if (
            file.is_file()
            and any(
                keyword in file.name.casefold()
                for keyword in IMPORTANT_KEYWORDS
            )
        )
    ],
    key=lambda file: normalize_name(str(file))
)

print("\nKeyword-matched files:")
print("-" * 80)

if keyword_candidates:

    for index, file in enumerate(
        keyword_candidates,
        start=1
    ):
        print(
            f"{index:03d} | "
            f"{file.relative_to(MOUTH_ROOT)}"
        )

else:
    print("No keyword-matched files were found.")


# ------------------------------------------------------------
# Find unusual image filename structure
# Cell 6 showed one file with four underscore tokens
# ------------------------------------------------------------
unusual_image_names = [
    path
    for path in all_dataset_paths
    if len(path.stem.split("_")) != 6
]

print("\nUnusual image filename structures:")
print("-" * 80)

if unusual_image_names:

    for path in unusual_image_names:
        print(
            path.relative_to(MOUTH_ROOT),
            "| tokens:",
            len(path.stem.split("_"))
        )

else:
    print("No unusual image filenames were found.")


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("\n" + "-" * 80)

if metadata_candidates:
    print(
        "Metadata status: CANDIDATE FILES FOUND"
    )
    print(
        "The files must be inspected before creating "
        "a new manifest."
    )

else:
    print(
        "Metadata status: NO EXISTING MANIFEST FOUND"
    )
    print(
        "A limited image-level manifest can be created, "
        "but source_video must not be invented."
    )

print("=" * 80)
print("CELL 7 COMPLETED")


METADATA AND SPLIT MANIFEST SEARCH
Search root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Ağız
Metadata-like files found: 3045

Candidate metadata files:
--------------------------------------------------------------------------------
001 | HOG_LBP_KAZE_Results/feature_chunks/features_0000_0249.npz | 2897.42 KB
002 | HOG_LBP_KAZE_Results/feature_chunks/features_0250_0499.npz | 2876.42 KB
003 | HOG_LBP_KAZE_Results/feature_chunks/features_0500_0749.npz | 2830.63 KB
004 | HOG_LBP_KAZE_Results/feature_chunks/features_0750_0999.npz | 2860.23 KB
005 | HOG_LBP_KAZE_Results/feature_chunks/features_1000_1249.npz | 2869.67 KB
006 | HOG_LBP_KAZE_Results/feature_chunks/features_1250_1499.npz | 2862.87 KB
007 | HOG_LBP_KAZE_Results/feature_chunks/features_1500_1749.npz | 2922.29 KB
008 | HOG_LBP_KAZE_Results/feature_chunks/features_1750_1999.npz | 2918.16 KB
009 | HOG_LBP_KAZE_Results/feature_chunks/features_2000_2249.npz | 2933.01 KB
010 | HOG_LBP_KAZE_Results/f

In [10]:
# ============================================================
# CELL 8 — MASTER METADATA AND SOURCE VIDEO AUDIT
# ============================================================

import pandas as pd
from pathlib import Path
import json

print("\n" + "=" * 85)
print("MASTER METADATA AND SOURCE VIDEO AUDIT")
print("=" * 85)

# ------------------------------------------------------------
# Important existing metadata files
# ------------------------------------------------------------
MASTER_METADATA_PATH = (
    DATASET_ROOT /
    "metadata.csv"
)

KAZE_METADATA_PATH = (
    MOUTH_ROOT /
    "HOG_LBP_KAZE_Results" /
    "mouth_hog_lbp_kaze_metadata.csv"
)

XCEPTION_LEAKAGE_PATH = (
    MOUTH_ROOT /
    "xception_experiment" /
    "results" /
    "source_video_leakage_check.csv"
)

PROCESSING_SUMMARY_PATH = (
    DATASET_ROOT /
    "processing_summary.json"
)


# ------------------------------------------------------------
# Verify required files
# ------------------------------------------------------------
required_audit_files = {
    "Master metadata": MASTER_METADATA_PATH,
    "KAZE metadata": KAZE_METADATA_PATH,
    "Xception leakage check": XCEPTION_LEAKAGE_PATH,
    "Processing summary": PROCESSING_SUMMARY_PATH
}

print("Existing audit files:")
print("-" * 85)

for file_name, file_path in required_audit_files.items():
    status = "FOUND" if file_path.exists() else "NOT FOUND"

    print(
        f"{file_name:28}: "
        f"{status:9} | {file_path}"
    )


assert MASTER_METADATA_PATH.exists(), (
    f"Master metadata was not found:\n"
    f"{MASTER_METADATA_PATH}"
)


# ------------------------------------------------------------
# Read master metadata
# ------------------------------------------------------------
master_metadata = pd.read_csv(
    MASTER_METADATA_PATH
)

print("\n" + "-" * 85)
print("MASTER METADATA SUMMARY")
print("-" * 85)

print("Shape   :", master_metadata.shape)
print("Columns :", master_metadata.columns.tolist())

print("\nData types:")
print(master_metadata.dtypes.to_string())

print("\nFirst five rows:")
print(
    master_metadata.head().to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Search for important standard columns
# ------------------------------------------------------------
column_lookup = {
    column.casefold(): column
    for column in master_metadata.columns
}

COLUMN_CANDIDATES = {
    "sample_id": [
        "sample_id",
        "id",
        "roi_id"
    ],

    "source_video": [
        "source_video",
        "video",
        "video_id",
        "source",
        "source_path",
        "video_path"
    ],

    "frame_index": [
        "frame_index",
        "frame",
        "frame_id",
        "frame_number"
    ],

    "face_index": [
        "face_index",
        "face_id"
    ],

    "label": [
        "label",
        "class",
        "target"
    ],

    "split": [
        "split",
        "subset",
        "partition"
    ],

    "output_path": [
        "output_path",
        "mouth_path",
        "roi_path",
        "image_path",
        "path"
    ],

    "status": [
        "status",
        "state"
    ],

    "skip_reason": [
        "skip_reason",
        "reason"
    ],

    "sha256": [
        "sha256",
        "hash",
        "file_hash"
    ]
}


def find_existing_column(candidate_names):
    for candidate_name in candidate_names:
        normalized_candidate = candidate_name.casefold()

        if normalized_candidate in column_lookup:
            return column_lookup[normalized_candidate]

    return None


detected_columns = {
    standard_name: find_existing_column(
        candidate_names
    )
    for standard_name, candidate_names
    in COLUMN_CANDIDATES.items()
}

print("\nDetected standard columns:")
print("-" * 85)

for standard_name, actual_column in detected_columns.items():

    print(
        f"{standard_name:15}: "
        f"{actual_column if actual_column else 'NOT FOUND'}"
    )


# ------------------------------------------------------------
# Display unique values for key columns
# ------------------------------------------------------------
for standard_name in [
    "label",
    "split",
    "status"
]:
    actual_column = detected_columns[standard_name]

    if actual_column is not None:

        value_counts = (
            master_metadata[actual_column]
            .astype(str)
            .value_counts(dropna=False)
        )

        print(
            f"\nValue counts — {actual_column}:"
        )

        print(value_counts.to_string())


# ------------------------------------------------------------
# Source video summary
# ------------------------------------------------------------
source_video_column = detected_columns[
    "source_video"
]

if source_video_column is not None:

    source_series = (
        master_metadata[source_video_column]
        .dropna()
        .astype(str)
    )

    print("\nSource video information:")
    print(
        "Non-null rows       :",
        len(source_series)
    )
    print(
        "Unique source videos:",
        source_series.nunique()
    )

    print("\nExample source videos:")

    for source_video in source_series.unique()[:10]:
        print(" -", source_video)

else:
    print(
        "\nWARNING: A source-video column was not "
        "detected automatically."
    )


# ------------------------------------------------------------
# Existing Xception leakage-check file
# ------------------------------------------------------------
if XCEPTION_LEAKAGE_PATH.exists():

    xception_leakage_df = pd.read_csv(
        XCEPTION_LEAKAGE_PATH
    )

    print("\n" + "-" * 85)
    print("EXISTING XCEPTION LEAKAGE CHECK")
    print("-" * 85)

    print("Shape   :", xception_leakage_df.shape)
    print(
        "Columns :",
        xception_leakage_df.columns.tolist()
    )

    print(
        xception_leakage_df.to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# Processing summary
# ------------------------------------------------------------
if PROCESSING_SUMMARY_PATH.exists():

    with open(
        PROCESSING_SUMMARY_PATH,
        "r",
        encoding="utf-8"
    ) as summary_file:
        processing_summary = json.load(
            summary_file
        )

    print("\n" + "-" * 85)
    print("PROCESSING SUMMARY")
    print("-" * 85)

    print(
        json.dumps(
            processing_summary,
            indent=2,
            ensure_ascii=False
        )[:5000]
    )


print("\n" + "=" * 85)
print("CELL 8 COMPLETED")
print("=" * 85)


MASTER METADATA AND SOURCE VIDEO AUDIT
Existing audit files:
-------------------------------------------------------------------------------------
Master metadata             : FOUND     | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output/metadata.csv
KAZE metadata               : FOUND     | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Ağız/HOG_LBP_KAZE_Results/mouth_hog_lbp_kaze_metadata.csv
Xception leakage check      : FOUND     | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Ağız/xception_experiment/results/source_video_leakage_check.csv
Processing summary          : FOUND     | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output/processing_summary.json

-------------------------------------------------------------------------------------
MASTER METADATA SUMMARY
------------------------------------------------------------------

In [11]:
# ============================================================
# CELL 9 — STANDARD MANIFEST, HASH AND LEAKAGE QUALITY GATES
# ============================================================

import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

print("\n" + "=" * 90)
print("STANDARD MANIFEST, HASH AND LEAKAGE QUALITY GATES")
print("=" * 90)


# ------------------------------------------------------------
# Convert the existing Run ID to the SSOT format
#
# Old:
# YYYYMMDD_HHMMSS_densenet121_mouth_seed42
#
# Standard:
# YYYYMMDD_HHMM_mouth_densenet121_seed42
# ------------------------------------------------------------
run_id_match = re.match(
    r"(\d{8})_(\d{4})\d{2}_densenet121_mouth_seed42",
    RUN_ID
)

if run_id_match is None:
    raise ValueError(
        f"Unexpected original Run ID format: {RUN_ID}"
    )

run_date = run_id_match.group(1)
run_time = run_id_match.group(2)

STANDARD_RUN_ID = (
    f"{run_date}_{run_time}_"
    f"mouth_densenet121_seed42"
)

# From this point onward, use the standard Run ID
RUN_ID = STANDARD_RUN_ID

RUN_ROOT = (
    RESULTS_ROOT /
    "DenseNet121_Mouth_Results" /
    RUN_ID
)


# ------------------------------------------------------------
# Standard experiment directories
# ------------------------------------------------------------
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
LOG_DIR = RUN_ROOT / "logs"
METRICS_DIR = RUN_ROOT / "metrics"
PREDICTION_DIR = RUN_ROOT / "predictions"
FIGURE_DIR = RUN_ROOT / "figures"
METADATA_DIR = RUN_ROOT / "metadata"

for directory in [
    CHECKPOINT_DIR,
    LOG_DIR,
    METRICS_DIR,
    PREDICTION_DIR,
    FIGURE_DIR,
    METADATA_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# Compatibility aliases for later cells
MODEL_DIR = CHECKPOINT_DIR
TABLE_DIR = METRICS_DIR


# ------------------------------------------------------------
# Metadata accounting equality
# ------------------------------------------------------------
status_normalized = (
    master_metadata["status"]
    .astype(str)
    .str.upper()
    .str.strip()
)

success_count = int(
    (status_normalized == "SUCCESS").sum()
)

skipped_count = int(
    (status_normalized == "SKIPPED").sum()
)

error_count = int(
    (status_normalized == "ERROR").sum()
)

total_metadata_records = len(
    master_metadata
)

assert total_metadata_records == (
    success_count +
    skipped_count +
    error_count
), (
    "Metadata accounting equality failed.\n"
    f"Total   : {total_metadata_records}\n"
    f"Success : {success_count}\n"
    f"Skipped : {skipped_count}\n"
    f"Error   : {error_count}"
)

assert success_count == 2987, (
    f"Expected 2987 successful ROI records, "
    f"found {success_count}."
)

assert skipped_count == 111, (
    f"Expected 111 skipped records, "
    f"found {skipped_count}."
)

assert error_count == 0, (
    f"Expected zero error records, "
    f"found {error_count}."
)


# ------------------------------------------------------------
# Keep only successful mouth ROI metadata
# ------------------------------------------------------------
successful_metadata = (
    master_metadata.loc[
        status_normalized == "SUCCESS"
    ]
    .copy()
    .reset_index(drop=True)
)

successful_metadata["mouth_filename"] = (
    successful_metadata["mouth_path"]
    .astype(str)
    .map(lambda value: Path(value).name)
)

assert successful_metadata[
    "sample_id"
].notna().all(), (
    "Missing sample_id exists in successful metadata."
)

assert successful_metadata[
    "sample_id"
].is_unique, (
    "Duplicate sample_id exists in successful metadata."
)

assert successful_metadata[
    "mouth_filename"
].notna().all(), (
    "Missing mouth filename exists."
)

assert successful_metadata[
    "mouth_filename"
].is_unique, (
    "Duplicate mouth filename exists in successful metadata."
)


# ------------------------------------------------------------
# Build lookup from existing metadata
# ------------------------------------------------------------
metadata_by_filename = (
    successful_metadata
    .set_index("mouth_filename")
)

current_image_records = []

for split_name in ["train", "val", "test"]:

    image_paths = dataset_records[
        split_name
    ]["paths"]

    image_labels = dataset_records[
        split_name
    ]["labels"]

    for image_path, numeric_label in zip(
        image_paths,
        image_labels
    ):
        current_path = Path(image_path)
        class_name = (
            "fake"
            if int(numeric_label) == 1
            else "real"
        )

        current_image_records.append(
            {
                "filename": current_path.name,
                "output_path": str(current_path),
                "label_from_folder": class_name,
                "split_from_folder": split_name
            }
        )

current_images_df = pd.DataFrame(
    current_image_records
)

assert len(current_images_df) == 2987, (
    "Current image accounting failed."
)

assert current_images_df[
    "filename"
].is_unique, (
    "Current mouth image filenames are not unique."
)


# ------------------------------------------------------------
# Match current image files with master metadata
# ------------------------------------------------------------
missing_from_metadata = sorted(
    set(current_images_df["filename"]) -
    set(metadata_by_filename.index)
)

missing_from_disk_listing = sorted(
    set(metadata_by_filename.index) -
    set(current_images_df["filename"])
)

assert not missing_from_metadata, (
    "Some current images are missing from metadata:\n"
    f"{missing_from_metadata[:20]}"
)

assert not missing_from_disk_listing, (
    "Some successful metadata rows have no current image:\n"
    f"{missing_from_disk_listing[:20]}"
)


matched_metadata = (
    metadata_by_filename
    .loc[current_images_df["filename"]]
    .reset_index()
)

# Add the currently valid Drive path
matched_metadata["current_output_path"] = (
    current_images_df["output_path"].values
)

matched_metadata["label_from_folder"] = (
    current_images_df["label_from_folder"].values
)

matched_metadata["split_from_folder"] = (
    current_images_df["split_from_folder"].values
)


# ------------------------------------------------------------
# Verify folder labels and splits against metadata
# ------------------------------------------------------------
metadata_labels = (
    matched_metadata["label"]
    .astype(str)
    .str.lower()
    .str.strip()
)

metadata_splits = (
    matched_metadata["split"]
    .astype(str)
    .str.lower()
    .str.strip()
)

folder_labels = (
    matched_metadata["label_from_folder"]
    .astype(str)
    .str.lower()
    .str.strip()
)

folder_splits = (
    matched_metadata["split_from_folder"]
    .astype(str)
    .str.lower()
    .str.strip()
)

assert (
    metadata_labels.values ==
    folder_labels.values
).all(), (
    "Folder labels and metadata labels do not match."
)

assert (
    metadata_splits.values ==
    folder_splits.values
).all(), (
    "Folder splits and metadata splits do not match."
)


# ------------------------------------------------------------
# Frame index extraction from frame_stem
# Example: fake_train_01130 -> 1130
# ------------------------------------------------------------
def extract_frame_index(frame_stem):
    match = re.search(
        r"(\d+)$",
        str(frame_stem)
    )

    if match is None:
        return np.nan

    return int(match.group(1))


matched_metadata["frame_index"] = (
    matched_metadata["frame_stem"]
    .map(extract_frame_index)
)

assert matched_metadata[
    "frame_index"
].notna().all(), (
    "Frame index extraction failed."
)


# ------------------------------------------------------------
# SHA-256 hashing
# This can take a little time because files are on Drive.
# ------------------------------------------------------------
def calculate_sha256(file_path):
    hash_object = hashlib.sha256()

    with open(file_path, "rb") as file_stream:
        while True:
            data_chunk = file_stream.read(
                1024 * 1024
            )

            if not data_chunk:
                break

            hash_object.update(data_chunk)

    return hash_object.hexdigest()


print("\nCalculating SHA-256 hashes for 2,987 images...")

sha256_values = []

for index, file_path in enumerate(
    matched_metadata["current_output_path"],
    start=1
):
    sha256_values.append(
        calculate_sha256(file_path)
    )

    if (
        index % 500 == 0
        or index == len(matched_metadata)
    ):
        print(
            f"Hashed: {index} / "
            f"{len(matched_metadata)}"
        )

matched_metadata["sha256"] = sha256_values


# ------------------------------------------------------------
# Build the standard model manifest
# ------------------------------------------------------------
model_manifest = pd.DataFrame(
    {
        "sample_id": matched_metadata["sample_id"],
        "source_video": matched_metadata["video_id"],
        "frame_index": (
            matched_metadata["frame_index"]
            .astype(int)
        ),
        "face_index": (
            matched_metadata["face_id"]
            .fillna(0)
            .astype(int)
        ),
        "roi_state": "not_recorded",
        "label": metadata_labels,
        "split": metadata_splits,
        "status": "SUCCESS",
        "skip_reason": "",
        "sha256": matched_metadata["sha256"],
        "output_path": (
            matched_metadata["current_output_path"]
        ),
        "run_id": RUN_ID,
        "source_frame": (
            matched_metadata["source_frame"]
        ),
        "mouth_width_px": (
            matched_metadata["mouth_width_px"]
        ),
        "mouth_height_px": (
            matched_metadata["mouth_height_px"]
        )
    }
)


# ------------------------------------------------------------
# Manifest quality gates
# ------------------------------------------------------------
REQUIRED_MANIFEST_COLUMNS = [
    "sample_id",
    "source_video",
    "frame_index",
    "face_index",
    "roi_state",
    "label",
    "split",
    "status",
    "skip_reason",
    "sha256",
    "output_path",
    "run_id"
]

missing_required_columns = (
    set(REQUIRED_MANIFEST_COLUMNS) -
    set(model_manifest.columns)
)

assert not missing_required_columns, (
    "Required manifest columns are missing:\n"
    f"{sorted(missing_required_columns)}"
)

assert model_manifest[
    "sample_id"
].is_unique, (
    "Duplicate sample_id detected."
)

assert model_manifest[
    "output_path"
].notna().all(), (
    "Missing output_path detected."
)

assert model_manifest[
    "sha256"
].notna().all(), (
    "Missing SHA-256 detected."
)

assert model_manifest[
    "source_video"
].notna().all(), (
    "Missing source_video detected."
)

assert all(
    Path(path).exists()
    for path in model_manifest["output_path"]
), (
    "A SUCCESS image is missing from disk."
)


# ------------------------------------------------------------
# Video/source leakage checks
# ------------------------------------------------------------
train_video_ids = set(
    model_manifest.loc[
        model_manifest["split"] == "train",
        "source_video"
    ]
)

val_video_ids = set(
    model_manifest.loc[
        model_manifest["split"] == "val",
        "source_video"
    ]
)

test_video_ids = set(
    model_manifest.loc[
        model_manifest["split"] == "test",
        "source_video"
    ]
)

video_overlap_counts = {
    "train_vs_val": len(
        train_video_ids & val_video_ids
    ),
    "train_vs_test": len(
        train_video_ids & test_video_ids
    ),
    "val_vs_test": len(
        val_video_ids & test_video_ids
    )
}

assert video_overlap_counts[
    "train_vs_val"
] == 0, (
    "Source-video leakage between train and validation."
)

assert video_overlap_counts[
    "train_vs_test"
] == 0, (
    "Source-video leakage between train and test."
)

assert video_overlap_counts[
    "val_vs_test"
] == 0, (
    "Source-video leakage between validation and test."
)


# ------------------------------------------------------------
# Content-hash leakage checks
# ------------------------------------------------------------
train_hashes = set(
    model_manifest.loc[
        model_manifest["split"] == "train",
        "sha256"
    ]
)

val_hashes = set(
    model_manifest.loc[
        model_manifest["split"] == "val",
        "sha256"
    ]
)

test_hashes = set(
    model_manifest.loc[
        model_manifest["split"] == "test",
        "sha256"
    ]
)

hash_overlap_counts = {
    "train_vs_val": len(
        train_hashes & val_hashes
    ),
    "train_vs_test": len(
        train_hashes & test_hashes
    ),
    "val_vs_test": len(
        val_hashes & test_hashes
    )
}

assert hash_overlap_counts[
    "train_vs_val"
] == 0, (
    "Image-content leakage between train and validation."
)

assert hash_overlap_counts[
    "train_vs_test"
] == 0, (
    "Image-content leakage between train and test."
)

assert hash_overlap_counts[
    "val_vs_test"
] == 0, (
    "Image-content leakage between validation and test."
)


# ------------------------------------------------------------
# Save manifest and audit reports
# ------------------------------------------------------------
MANIFEST_PATH = (
    METADATA_DIR /
    "model_manifest.csv"
)

LEAKAGE_REPORT_PATH = (
    METADATA_DIR /
    "leakage_check.csv"
)

ACCOUNTING_REPORT_PATH = (
    METADATA_DIR /
    "accounting_summary.json"
)

model_manifest.to_csv(
    MANIFEST_PATH,
    index=False
)

leakage_report = pd.DataFrame(
    [
        {
            "comparison": comparison,
            "source_video_overlap": (
                video_overlap_counts[comparison]
            ),
            "sha256_overlap": (
                hash_overlap_counts[comparison]
            )
        }
        for comparison in [
            "train_vs_val",
            "train_vs_test",
            "val_vs_test"
        ]
    ]
)

leakage_report.to_csv(
    LEAKAGE_REPORT_PATH,
    index=False
)

accounting_summary = {
    "run_id": RUN_ID,
    "total_metadata_records": (
        total_metadata_records
    ),
    "success_count": success_count,
    "skipped_count": skipped_count,
    "error_count": error_count,
    "model_image_count": len(model_manifest),
    "video_overlap_counts": video_overlap_counts,
    "sha256_overlap_counts": hash_overlap_counts
}

with open(
    ACCOUNTING_REPORT_PATH,
    "w",
    encoding="utf-8"
) as summary_file:
    json.dump(
        accounting_summary,
        summary_file,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------
print("\n" + "-" * 90)
print("QUALITY-GATE RESULTS")
print("-" * 90)

print(
    "Metadata accounting :",
    f"{total_metadata_records} = "
    f"{success_count} SUCCESS + "
    f"{skipped_count} SKIPPED + "
    f"{error_count} ERROR"
)

print("Model images       :", len(model_manifest))
print(
    "Unique sample IDs  :",
    model_manifest["sample_id"].nunique()
)

print(
    "Unique source IDs  :",
    model_manifest["source_video"].nunique()
)

print("\nSource-video overlaps:")
print(video_overlap_counts)

print("\nSHA-256 overlaps:")
print(hash_overlap_counts)

print("\nStandard Run ID :", RUN_ID)
print("Run root        :", RUN_ROOT)
print("Manifest        :", MANIFEST_PATH)
print("Leakage report  :", LEAKAGE_REPORT_PATH)

print("\nManifest split counts:")
print(
    pd.crosstab(
        model_manifest["split"],
        model_manifest["label"]
    ).to_string()
)

print("=" * 90)
print("METADATA ACCOUNTING       : PASS")
print("IMAGE-METADATA MATCH      : PASS")
print("SOURCE-VIDEO LEAKAGE TEST : PASS")
print("SHA-256 LEAKAGE TEST      : PASS")
print("CELL 9 COMPLETED")
print("=" * 90)


STANDARD MANIFEST, HASH AND LEAKAGE QUALITY GATES

Calculating SHA-256 hashes for 2,987 images...
Hashed: 500 / 2987
Hashed: 1000 / 2987
Hashed: 1500 / 2987
Hashed: 2000 / 2987
Hashed: 2500 / 2987
Hashed: 2987 / 2987

------------------------------------------------------------------------------------------
QUALITY-GATE RESULTS
------------------------------------------------------------------------------------------
Metadata accounting : 3098 = 2987 SUCCESS + 111 SKIPPED + 0 ERROR
Model images       : 2987
Unique sample IDs  : 2987
Unique source IDs  : 2889

Source-video overlaps:
{'train_vs_val': 0, 'train_vs_test': 0, 'val_vs_test': 0}

SHA-256 overlaps:
{'train_vs_val': 0, 'train_vs_test': 0, 'val_vs_test': 0}

Standard Run ID : 20260808_0856_mouth_densenet121_seed42
Run root        : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42
Manifest        : /content/drive/MyDrive/AISC Dee

In [12]:
# ============================================================
# CELL 10 — BUILD FROZEN DENSENET121 MODEL
# ============================================================

import os
import json
from pathlib import Path

import tensorflow as tf
import yaml

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dropout,
    Dense
)
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import (
    BinaryAccuracy,
    Precision,
    Recall,
    AUC
)

print("\n" + "=" * 85)
print("BUILDING FROZEN DENSENET121 MODEL")
print("=" * 85)


# ------------------------------------------------------------
# Mixed precision for T4 GPU
# ------------------------------------------------------------
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy(
    "mixed_float16"
)

print(
    "Mixed precision policy:",
    mixed_precision.global_policy()
)


# ------------------------------------------------------------
# Model hyperparameters
# ------------------------------------------------------------
MODEL_CONFIG = {
    "model_name": "DenseNet121",
    "weights": "imagenet",
    "include_top": False,
    "input_shape": [
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        3
    ],
    "global_pooling": (
        "GlobalAveragePooling2D"
    ),
    "dropout_rate": 0.40,
    "output_units": 1,
    "output_activation": "sigmoid",
    "output_dtype": "float32",
    "frozen_learning_rate": 1e-3,
    "finetune_learning_rate": 1e-5,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "label_mapping": {
        "real": 0,
        "fake": 1
    },
    "model_selection_metric": "val_auc",
    "model_selection_mode": "max"
}


# ------------------------------------------------------------
# Build ImageNet-pretrained DenseNet121 backbone
# ------------------------------------------------------------
print(
    "\nLoading ImageNet-pretrained "
    "DenseNet121 weights..."
)

base_model = DenseNet121(
    weights=MODEL_CONFIG["weights"],
    include_top=MODEL_CONFIG["include_top"],
    input_shape=tuple(
        MODEL_CONFIG["input_shape"]
    )
)

# Freeze the complete DenseNet121 backbone
base_model.trainable = False


# ------------------------------------------------------------
# Build binary classification model
# ------------------------------------------------------------
model_input = Input(
    shape=tuple(
        MODEL_CONFIG["input_shape"]
    ),
    name="mouth_roi_input"
)

# training=False keeps Batch Normalization layers
# in inference mode during frozen training.
features = base_model(
    model_input,
    training=False
)

features = GlobalAveragePooling2D(
    name="global_average_pooling"
)(features)

features = Dropout(
    rate=MODEL_CONFIG["dropout_rate"],
    seed=SEED,
    name="classification_dropout"
)(features)

model_output = Dense(
    units=MODEL_CONFIG["output_units"],
    activation=MODEL_CONFIG[
        "output_activation"
    ],
    dtype=MODEL_CONFIG[
        "output_dtype"
    ],
    name="real_fake_probability"
)(features)

model = Model(
    inputs=model_input,
    outputs=model_output,
    name="DenseNet121_Mouth_Deepfake"
)


# ------------------------------------------------------------
# Compile frozen-stage model
# ------------------------------------------------------------
frozen_optimizer = Adam(
    learning_rate=MODEL_CONFIG[
        "frozen_learning_rate"
    ]
)

model.compile(
    optimizer=frozen_optimizer,

    loss=BinaryCrossentropy(
        name="binary_crossentropy"
    ),

    metrics=[
        BinaryAccuracy(
            name="accuracy",
            threshold=0.5
        ),

        Precision(
            name="precision",
            thresholds=0.5
        ),

        Recall(
            name="recall",
            thresholds=0.5
        ),

        AUC(
            name="auc",
            curve="ROC"
        ),

        AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)


# ------------------------------------------------------------
# Parameter accounting
# ------------------------------------------------------------
total_parameters = model.count_params()

trainable_parameters = int(
    sum(
        tf.keras.backend.count_params(
            variable
        )
        for variable in model.trainable_weights
    )
)

non_trainable_parameters = (
    total_parameters -
    trainable_parameters
)

base_total_parameters = (
    base_model.count_params()
)

base_trainable_parameters = int(
    sum(
        tf.keras.backend.count_params(
            variable
        )
        for variable in base_model.trainable_weights
    )
)


# ------------------------------------------------------------
# Model assertions
# ------------------------------------------------------------
assert base_model.trainable is False, (
    "DenseNet121 backbone must be frozen."
)

assert base_trainable_parameters == 0, (
    "Frozen DenseNet121 contains trainable parameters."
)

assert model.output_shape == (
    None,
    1
), (
    f"Unexpected model output shape: "
    f"{model.output_shape}"
)

assert model.output.dtype == "float32", (
    "The sigmoid output must use float32 "
    "under mixed precision."
)


# ------------------------------------------------------------
# Forward-pass smoke check
# No model training is performed here.
# ------------------------------------------------------------
forward_test_batch = sample_images[:2]

forward_test_predictions = model(
    forward_test_batch,
    training=False
)

assert forward_test_predictions.shape == (
    2,
    1
), (
    "Forward-pass output shape is incorrect."
)

assert bool(
    tf.reduce_all(
        tf.math.is_finite(
            forward_test_predictions
        )
    )
), (
    "NaN or Inf was detected in model outputs."
)

assert bool(
    tf.reduce_all(
        forward_test_predictions >= 0.0
    )
), (
    "Prediction below zero was detected."
)

assert bool(
    tf.reduce_all(
        forward_test_predictions <= 1.0
    )
), (
    "Prediction above one was detected."
)


# ------------------------------------------------------------
# Standard output paths
# ------------------------------------------------------------
BEST_MODEL_PATH = (
    CHECKPOINT_DIR /
    "best.keras"
)

LAST_MODEL_PATH = (
    CHECKPOINT_DIR /
    "last.keras"
)

BACKUP_DIR = (
    CHECKPOINT_DIR /
    "training_backup"
)

MODEL_SUMMARY_PATH = (
    METRICS_DIR /
    "model_summary.txt"
)

RESOLVED_CONFIG_PATH = (
    RUN_ROOT /
    "config_resolved.yaml"
)


# ------------------------------------------------------------
# Save model summary
# ------------------------------------------------------------
summary_lines = []

model.summary(
    print_fn=lambda line: (
        summary_lines.append(line)
    )
)

with open(
    MODEL_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as summary_file:
    summary_file.write(
        "\n".join(summary_lines)
    )


# ------------------------------------------------------------
# Save resolved experiment configuration atomically
# ------------------------------------------------------------
resolved_config = {
    "run_id": RUN_ID,

    "task": (
        "Mouth ROI Deepfake "
        "Binary Classification"
    ),

    "framework": "TensorFlow / Keras",

    "tensorflow_version": tf.__version__,

    "mixed_precision_policy": str(
        mixed_precision.global_policy()
    ),

    "dataset": {
        "dataset_root": str(DATASET_ROOT),
        "manifest_path": str(
            MANIFEST_PATH
        ),
        "train_count": 2389,
        "validation_count": 296,
        "test_count": 302,
        "image_height": IMAGE_HEIGHT,
        "image_width": IMAGE_WIDTH,
        "channels": 3
    },

    "model": MODEL_CONFIG,

    "parameters": {
        "total": total_parameters,
        "trainable_frozen_stage": (
            trainable_parameters
        ),
        "non_trainable_frozen_stage": (
            non_trainable_parameters
        ),
        "base_model_total": (
            base_total_parameters
        ),
        "base_model_trainable": (
            base_trainable_parameters
        )
    },

    "output_paths": {
        "run_root": str(RUN_ROOT),
        "best_model": str(
            BEST_MODEL_PATH
        ),
        "last_model": str(
            LAST_MODEL_PATH
        ),
        "backup_directory": str(
            BACKUP_DIR
        )
    }
}

temporary_config_path = (
    RESOLVED_CONFIG_PATH.with_suffix(
        ".yaml.tmp"
    )
)

with open(
    temporary_config_path,
    "w",
    encoding="utf-8"
) as config_file:
    yaml.safe_dump(
        resolved_config,
        config_file,
        allow_unicode=True,
        sort_keys=False
    )

# Atomic replacement
os.replace(
    temporary_config_path,
    RESOLVED_CONFIG_PATH
)


# ------------------------------------------------------------
# Print model information
# ------------------------------------------------------------
print("\n" + "-" * 85)
print("MODEL PARAMETER SUMMARY")
print("-" * 85)

print(
    f"Total parameters        : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters    : "
    f"{trainable_parameters:,}"
)

print(
    f"Non-trainable parameters: "
    f"{non_trainable_parameters:,}"
)

print(
    f"Backbone parameters     : "
    f"{base_total_parameters:,}"
)

print(
    f"Backbone trainable      : "
    f"{base_trainable_parameters:,}"
)

print("\nForward-test predictions:")

print(
    forward_test_predictions
    .numpy()
    .reshape(-1)
    .tolist()
)

print("\nModel summary :", MODEL_SUMMARY_PATH)
print("Config file  :", RESOLVED_CONFIG_PATH)
print("Best model   :", BEST_MODEL_PATH)
print("Last model   :", LAST_MODEL_PATH)

print("=" * 85)
print("DENSENET121 BACKBONE     : FROZEN")
print("MIXED PRECISION           : ENABLED")
print("FORWARD PASS TEST         : PASS")
print("MODEL CONFIGURATION       : SAVED")
print("CELL 10 COMPLETED")
print("=" * 85)


BUILDING FROZEN DENSENET121 MODEL
Mixed precision policy: <DTypePolicy "mixed_float16">

Loading ImageNet-pretrained DenseNet121 weights...
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step



-------------------------------------------------------------------------------------
MODEL PARAMETER SUMMARY
-------------------------------------------------------------------------------------
Total parameters        : 7,038,529
Trainable parameters    : 1,025
Non-trainable parameters: 7,037,504
Backbone parameters     : 7,037,504
Backbone trainable      : 0

Forward-test predictions:
[0.28032180666923523, 0.3444984555244446]

Model summary : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/metrics/model_summary.txt
Config file  : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/config_resolved.yaml
Best model   : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/checkpoints/best.keras

In [13]:
# ============================================================
# CELL 11 — TWO-BATCH TRAINING AND CHECKPOINT SMOKE TEST
# ============================================================

import os
from pathlib import Path

import numpy as np
import tensorflow as tf

print("\n" + "=" * 90)
print("TWO-BATCH TRAINING AND CHECKPOINT SMOKE TEST")
print("=" * 90)


# ------------------------------------------------------------
# Smoke-test checkpoint paths
# ------------------------------------------------------------
SMOKE_CHECKPOINT_PATH = (
    CHECKPOINT_DIR /
    "smoke_test_checkpoint.keras"
)

SMOKE_TEMP_PATH = (
    CHECKPOINT_DIR /
    "smoke_test_checkpoint.tmp.keras"
)


# ------------------------------------------------------------
# Keep the original untrained classification-head weights
# The smoke test must not affect the real experiment.
# ------------------------------------------------------------
initial_model_weights = model.get_weights()

initial_optimizer_iterations = int(
    model.optimizer.iterations.numpy()
)

print(
    "Initial optimizer iterations:",
    initial_optimizer_iterations
)


# ------------------------------------------------------------
# Obtain exactly two training batches
# ------------------------------------------------------------
smoke_batches = list(
    train_dataset.take(2)
)

assert len(smoke_batches) == 2, (
    "Two training batches could not be loaded."
)

for batch_index, (
    batch_images,
    batch_labels
) in enumerate(
    smoke_batches,
    start=1
):
    print(
        f"Batch {batch_index} shape:",
        batch_images.shape,
        batch_labels.shape
    )

    assert batch_images.shape[1:] == (
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        3
    )

    assert bool(
        tf.reduce_all(
            tf.math.is_finite(
                batch_images
            )
        )
    ), (
        f"NaN or Inf found in batch "
        f"{batch_index} images."
    )

    assert bool(
        tf.reduce_all(
            tf.math.is_finite(
                batch_labels
            )
        )
    ), (
        f"NaN or Inf found in batch "
        f"{batch_index} labels."
    )


# ------------------------------------------------------------
# Explicit forward and gradient numerical test
# No optimizer update is performed in this section.
# ------------------------------------------------------------
gradient_test_images = smoke_batches[0][0]
gradient_test_labels = smoke_batches[0][1]

gradient_test_labels = tf.reshape(
    gradient_test_labels,
    shape=(-1, 1)
)

gradient_loss_function = (
    tf.keras.losses.BinaryCrossentropy()
)

with tf.GradientTape() as tape:

    gradient_predictions = model(
        gradient_test_images,
        training=True
    )

    gradient_loss = gradient_loss_function(
        gradient_test_labels,
        gradient_predictions
    )

gradients = tape.gradient(
    gradient_loss,
    model.trainable_variables
)

assert bool(
    tf.math.is_finite(
        gradient_loss
    )
), (
    "NaN or Inf detected in smoke-test loss."
)

assert gradients, (
    "No gradients were generated."
)

assert all(
    gradient is not None
    for gradient in gradients
), (
    "A trainable variable has a missing gradient."
)

for gradient_index, gradient in enumerate(
    gradients
):
    assert bool(
        tf.reduce_all(
            tf.math.is_finite(
                gradient
            )
        )
    ), (
        f"NaN or Inf detected in gradient "
        f"{gradient_index}."
    )

print(
    "\nExplicit gradient-test loss:",
    float(gradient_loss.numpy())
)

print(
    "Gradient tensors checked:",
    len(gradients)
)


# ------------------------------------------------------------
# Perform exactly two optimizer update steps
# ------------------------------------------------------------
training_step_results = []

for batch_index, (
    batch_images,
    batch_labels
) in enumerate(
    smoke_batches,
    start=1
):

    batch_result = model.train_on_batch(
        batch_images,
        batch_labels,
        return_dict=True
    )

    clean_batch_result = {
        metric_name: float(metric_value)
        for metric_name, metric_value
        in batch_result.items()
    }

    assert all(
        np.isfinite(metric_value)
        for metric_value
        in clean_batch_result.values()
    ), (
        f"NaN or Inf detected after "
        f"training batch {batch_index}."
    )

    training_step_results.append(
        clean_batch_result
    )

    print(
        f"\nTraining batch {batch_index}:"
    )

    for metric_name, metric_value in (
        clean_batch_result.items()
    ):
        print(
            f"  {metric_name:12}: "
            f"{metric_value:.6f}"
        )


optimizer_iterations_after_training = int(
    model.optimizer.iterations.numpy()
)

assert optimizer_iterations_after_training == (
    initial_optimizer_iterations + 2
), (
    "Optimizer iteration count did not "
    "increase by exactly two."
)


# ------------------------------------------------------------
# Reference predictions and loss before saving
# Use validation data without augmentation.
# ------------------------------------------------------------
reference_images, reference_labels = next(
    iter(val_dataset)
)

reference_labels_2d = tf.reshape(
    reference_labels,
    shape=(-1, 1)
)

predictions_before_save = model(
    reference_images,
    training=False
).numpy()

loss_before_save = float(
    gradient_loss_function(
        reference_labels_2d,
        predictions_before_save
    ).numpy()
)

assert np.isfinite(
    predictions_before_save
).all()

assert np.isfinite(
    loss_before_save
)


# ------------------------------------------------------------
# Full-state atomic checkpoint save
# The .keras file stores architecture, weights and optimizer.
# ------------------------------------------------------------
if SMOKE_TEMP_PATH.exists():
    SMOKE_TEMP_PATH.unlink()

model.save(
    SMOKE_TEMP_PATH,
    include_optimizer=True
)

# Verify the temporary checkpoint before publishing it
temporary_loaded_model = (
    tf.keras.models.load_model(
        SMOKE_TEMP_PATH
    )
)

temporary_predictions = (
    temporary_loaded_model(
        reference_images,
        training=False
    )
    .numpy()
)

assert np.allclose(
    predictions_before_save,
    temporary_predictions,
    rtol=1e-6,
    atol=1e-7
), (
    "Temporary checkpoint prediction "
    "verification failed."
)

# Atomic publish
os.replace(
    SMOKE_TEMP_PATH,
    SMOKE_CHECKPOINT_PATH
)


# ------------------------------------------------------------
# Reload the published checkpoint from disk
# ------------------------------------------------------------
reloaded_smoke_model = (
    tf.keras.models.load_model(
        SMOKE_CHECKPOINT_PATH
    )
)

predictions_after_reload = (
    reloaded_smoke_model(
        reference_images,
        training=False
    )
    .numpy()
)

loss_after_reload = float(
    gradient_loss_function(
        reference_labels_2d,
        predictions_after_reload
    ).numpy()
)

reloaded_optimizer_iterations = int(
    reloaded_smoke_model
    .optimizer
    .iterations
    .numpy()
)


# ------------------------------------------------------------
# Checkpoint equivalence assertions
# ------------------------------------------------------------
assert np.allclose(
    predictions_before_save,
    predictions_after_reload,
    rtol=1e-6,
    atol=1e-7
), (
    "Predictions changed after "
    "checkpoint reload."
)

assert np.isclose(
    loss_before_save,
    loss_after_reload,
    rtol=1e-6,
    atol=1e-7
), (
    "Loss changed after checkpoint reload."
)

assert reloaded_optimizer_iterations == (
    optimizer_iterations_after_training
), (
    "Optimizer state was not restored."
)


# ------------------------------------------------------------
# Restore the original pre-smoke-test weights
# ------------------------------------------------------------
model.set_weights(
    initial_model_weights
)

# Reset the optimizer so the real training starts cleanly.
frozen_optimizer = tf.keras.optimizers.Adam(
    learning_rate=MODEL_CONFIG[
        "frozen_learning_rate"
    ]
)

model.compile(
    optimizer=frozen_optimizer,

    loss=tf.keras.losses.BinaryCrossentropy(
        name="binary_crossentropy"
    ),

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy",
            threshold=0.5
        ),

        tf.keras.metrics.Precision(
            name="precision",
            thresholds=0.5
        ),

        tf.keras.metrics.Recall(
            name="recall",
            thresholds=0.5
        ),

        tf.keras.metrics.AUC(
            name="auc",
            curve="ROC"
        ),

        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)

assert int(
    model.optimizer.iterations.numpy()
) == 0, (
    "The real-training optimizer did not "
    "restart from iteration zero."
)


# ------------------------------------------------------------
# Verify restored initial predictions are finite
# ------------------------------------------------------------
restored_predictions = model(
    reference_images,
    training=False
).numpy()

assert np.isfinite(
    restored_predictions
).all(), (
    "Restored model produced NaN or Inf."
)


# ------------------------------------------------------------
# Save smoke-test report
# ------------------------------------------------------------
SMOKE_REPORT_PATH = (
    METRICS_DIR /
    "smoke_test_report.json"
)

smoke_test_report = {
    "run_id": RUN_ID,
    "tested_batches": 2,
    "gradient_loss": float(
        gradient_loss.numpy()
    ),
    "gradient_tensor_count": len(
        gradients
    ),
    "training_step_results": (
        training_step_results
    ),
    "optimizer_iterations_before": (
        initial_optimizer_iterations
    ),
    "optimizer_iterations_after_two_batches": (
        optimizer_iterations_after_training
    ),
    "reloaded_optimizer_iterations": (
        reloaded_optimizer_iterations
    ),
    "loss_before_save": (
        loss_before_save
    ),
    "loss_after_reload": (
        loss_after_reload
    ),
    "prediction_max_absolute_difference": (
        float(
            np.max(
                np.abs(
                    predictions_before_save -
                    predictions_after_reload
                )
            )
        )
    ),
    "checkpoint_path": str(
        SMOKE_CHECKPOINT_PATH
    ),
    "real_training_optimizer_reset": True,
    "status": "PASS"
}

temporary_report_path = (
    SMOKE_REPORT_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_report_path,
    "w",
    encoding="utf-8"
) as report_file:
    json.dump(
        smoke_test_report,
        report_file,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    temporary_report_path,
    SMOKE_REPORT_PATH
)


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------
print("\n" + "-" * 90)
print("CHECKPOINT RELOAD RESULTS")
print("-" * 90)

print(
    "Loss before save       :",
    loss_before_save
)

print(
    "Loss after reload      :",
    loss_after_reload
)

print(
    "Maximum prediction diff:",
    smoke_test_report[
        "prediction_max_absolute_difference"
    ]
)

print(
    "Optimizer iterations   :",
    reloaded_optimizer_iterations
)

print(
    "Real optimizer reset   :",
    int(
        model.optimizer
        .iterations
        .numpy()
    )
)

print("Checkpoint :", SMOKE_CHECKPOINT_PATH)
print("Report     :", SMOKE_REPORT_PATH)

print("=" * 90)
print("TWO-BATCH TRAINING TEST : PASS")
print("GRADIENT NaN/Inf TEST   : PASS")
print("ATOMIC CHECKPOINT TEST  : PASS")
print("CHECKPOINT RELOAD TEST  : PASS")
print("REAL MODEL RESET        : PASS")
print("CELL 11 COMPLETED")
print("=" * 90)


TWO-BATCH TRAINING AND CHECKPOINT SMOKE TEST
Initial optimizer iterations: 0
Batch 1 shape: (16, 224, 224, 3) (16,)
Batch 2 shape: (16, 224, 224, 3) (16,)

Explicit gradient-test loss: 0.7775877714157104
Gradient tensors checked: 2

Training batch 1:
  accuracy    : 0.375000
  auc         : 0.468254
  loss        : 0.815901
  pr_auc      : 0.609688
  precision   : 0.428571
  recall      : 0.333333

Training batch 2:
  accuracy    : 0.500000
  auc         : 0.552734
  loss        : 0.783942
  pr_auc      : 0.533074
  precision   : 0.500000
  recall      : 0.375000

------------------------------------------------------------------------------------------
CHECKPOINT RELOAD RESULTS
------------------------------------------------------------------------------------------
Loss before save       : 0.5215162038803101
Loss after reload      : 0.5215162038803101
Maximum prediction diff: 0.0
Optimizer iterations   : 2
Real optimizer reset   : 0
Checkpoint : /content/drive/MyDrive/AISC DeepFake

In [14]:
# ============================================================
# CELL 12 — FROZEN-BACKBONE TRAINING
# ============================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

print("\n" + "=" * 90)
print("FROZEN-BACKBONE TRAINING")
print("=" * 90)


# ------------------------------------------------------------
# Frozen-stage configuration
# ------------------------------------------------------------
FROZEN_EPOCHS = 12

FROZEN_HISTORY_PATH = (
    METRICS_DIR /
    "frozen_training_history.csv"
)

FROZEN_LOG_PATH = (
    LOG_DIR /
    "frozen_training_log.csv"
)

CHECKPOINT_STATE_PATH = (
    CHECKPOINT_DIR /
    "checkpoint_state.json"
)


# ------------------------------------------------------------
# Atomic full-model checkpoint callback
# ------------------------------------------------------------
class AtomicModelCheckpoint(
    tf.keras.callbacks.Callback
):
    """
    Saves:
    - last.keras after every completed epoch
    - best.keras when validation AUC improves

    Each model is first written to a temporary file,
    reloaded for integrity verification and then
    atomically published with os.replace().
    """

    def __init__(
        self,
        best_model_path,
        last_model_path,
        state_path,
        monitor="val_auc",
        mode="max"
    ):
        super().__init__()

        self.best_model_path = Path(
            best_model_path
        )

        self.last_model_path = Path(
            last_model_path
        )

        self.state_path = Path(
            state_path
        )

        self.monitor = monitor
        self.mode = mode

        if mode == "max":
            self.best_value = -np.inf
        elif mode == "min":
            self.best_value = np.inf
        else:
            raise ValueError(
                "mode must be 'max' or 'min'."
            )

        # Recover the previous best value if this
        # experiment is resumed.
        if self.state_path.exists():

            with open(
                self.state_path,
                "r",
                encoding="utf-8"
            ) as state_file:
                previous_state = json.load(
                    state_file
                )

            if self.monitor in previous_state:
                self.best_value = float(
                    previous_state[
                        self.monitor
                    ]
                )

    def _is_improvement(self, current_value):

        if self.mode == "max":
            return current_value > self.best_value

        return current_value < self.best_value

    def _atomic_save_model(
        self,
        target_path
    ):
        target_path = Path(target_path)

        temporary_path = (
            target_path.parent /
            f"{target_path.stem}.tmp.keras"
        )

        if temporary_path.exists():
            temporary_path.unlink()

        # Save complete model including optimizer state
        self.model.save(
            temporary_path,
            include_optimizer=True
        )

        # Integrity verification
        verification_model = (
            tf.keras.models.load_model(
                temporary_path
            )
        )

        assert verification_model.output_shape == (
            None,
            1
        ), (
            "Checkpoint verification failed: "
            "unexpected output shape."
        )

        # Release temporary verification model
        del verification_model

        # Atomic publish
        os.replace(
            temporary_path,
            target_path
        )

    def _atomic_save_state(
        self,
        epoch,
        current_value
    ):
        state_data = {
            "run_id": RUN_ID,
            "completed_epoch": int(
                epoch + 1
            ),
            "monitor": self.monitor,
            self.monitor: float(
                self.best_value
            ),
            "latest_value": float(
                current_value
            ),
            "best_model_path": str(
                self.best_model_path
            ),
            "last_model_path": str(
                self.last_model_path
            )
        }

        temporary_state_path = (
            self.state_path.with_suffix(
                ".json.tmp"
            )
        )

        with open(
            temporary_state_path,
            "w",
            encoding="utf-8"
        ) as state_file:
            json.dump(
                state_data,
                state_file,
                indent=4,
                ensure_ascii=False
            )

        os.replace(
            temporary_state_path,
            self.state_path
        )

    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):
        logs = logs or {}

        if self.monitor not in logs:
            raise KeyError(
                f"Monitored metric not found: "
                f"{self.monitor}"
            )

        current_value = float(
            logs[self.monitor]
        )

        if not np.isfinite(current_value):
            raise FloatingPointError(
                f"{self.monitor} is NaN or Inf."
            )

        # Always save the most recently completed epoch
        self._atomic_save_model(
            self.last_model_path
        )

        improved = self._is_improvement(
            current_value
        )

        if improved:
            self.best_value = current_value

            self._atomic_save_model(
                self.best_model_path
            )

            print(
                f"\nAtomic best checkpoint updated: "
                f"{self.monitor}="
                f"{current_value:.6f}"
            )

        self._atomic_save_state(
            epoch=epoch,
            current_value=current_value
        )


# ------------------------------------------------------------
# Epoch-level finite-value quality gate
# ------------------------------------------------------------
class FiniteMetricsGuard(
    tf.keras.callbacks.Callback
):
    """
    Stops training immediately if an epoch metric
    contains NaN or Inf.
    """

    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):
        logs = logs or {}

        invalid_metrics = {
            metric_name: metric_value
            for metric_name, metric_value
            in logs.items()
            if not np.isfinite(metric_value)
        }

        if invalid_metrics:
            raise FloatingPointError(
                "NaN or Inf detected in metrics: "
                f"{invalid_metrics}"
            )


# ------------------------------------------------------------
# Training callbacks
# ------------------------------------------------------------
atomic_checkpoint_callback = (
    AtomicModelCheckpoint(
        best_model_path=BEST_MODEL_PATH,
        last_model_path=LAST_MODEL_PATH,
        state_path=CHECKPOINT_STATE_PATH,
        monitor="val_auc",
        mode="max"
    )
)

frozen_callbacks = [
    atomic_checkpoint_callback,

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=str(BACKUP_DIR),
        save_freq="epoch",
        delete_checkpoint=False
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(FROZEN_LOG_PATH),
        separator=",",
        append=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        min_delta=1e-4,
        restore_best_weights=False,
        verbose=1
    ),

    tf.keras.callbacks.TerminateOnNaN(),

    FiniteMetricsGuard()
]


# ------------------------------------------------------------
# Pre-training assertions
# ------------------------------------------------------------
assert base_model.trainable is False, (
    "DenseNet121 must remain frozen "
    "during stage 1."
)

assert int(
    model.optimizer.iterations.numpy()
) == 0, (
    "Frozen training must start from "
    "optimizer iteration zero."
)

frozen_trainable_parameters = int(
    sum(
        tf.keras.backend.count_params(
            variable
        )
        for variable in model.trainable_weights
    )
)

assert frozen_trainable_parameters == 1025, (
    "Unexpected frozen-stage trainable "
    f"parameter count: "
    f"{frozen_trainable_parameters}"
)


# ------------------------------------------------------------
# Start frozen-backbone training
# ------------------------------------------------------------
print(
    "Run ID                 :",
    RUN_ID
)

print(
    "Frozen epochs maximum  :",
    FROZEN_EPOCHS
)

print(
    "Trainable parameters   :",
    f"{frozen_trainable_parameters:,}"
)

print(
    "Selection metric       : val_auc"
)

print(
    "Best checkpoint        :",
    BEST_MODEL_PATH
)

print(
    "Last checkpoint        :",
    LAST_MODEL_PATH
)

print(
    "Recovery backup        :",
    BACKUP_DIR
)

print(
    "\nStarting frozen-backbone "
    "training...\n"
)

frozen_history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=FROZEN_EPOCHS,
    callbacks=frozen_callbacks,
    verbose=1
)


# ------------------------------------------------------------
# Convert training history to a table
# ------------------------------------------------------------
frozen_history_df = pd.DataFrame(
    frozen_history.history
)

frozen_history_df.insert(
    0,
    "epoch",
    np.arange(
        1,
        len(frozen_history_df) + 1
    )
)

frozen_history_df.insert(
    1,
    "stage",
    "frozen"
)


# ------------------------------------------------------------
# Save history atomically
# ------------------------------------------------------------
temporary_history_path = (
    FROZEN_HISTORY_PATH.with_suffix(
        ".csv.tmp"
    )
)

frozen_history_df.to_csv(
    temporary_history_path,
    index=False
)

os.replace(
    temporary_history_path,
    FROZEN_HISTORY_PATH
)


# ------------------------------------------------------------
# Verify checkpoints
# ------------------------------------------------------------
assert BEST_MODEL_PATH.exists(), (
    "Best model checkpoint was not created."
)

assert LAST_MODEL_PATH.exists(), (
    "Last model checkpoint was not created."
)

verified_best_model = (
    tf.keras.models.load_model(
        BEST_MODEL_PATH
    )
)

verified_last_model = (
    tf.keras.models.load_model(
        LAST_MODEL_PATH
    )
)

best_check_predictions = (
    verified_best_model(
        sample_images[:2],
        training=False
    )
    .numpy()
)

last_check_predictions = (
    verified_last_model(
        sample_images[:2],
        training=False
    )
    .numpy()
)

assert np.isfinite(
    best_check_predictions
).all()

assert np.isfinite(
    last_check_predictions
).all()


# ------------------------------------------------------------
# Frozen-stage summary
# ------------------------------------------------------------
best_frozen_epoch_index = int(
    frozen_history_df[
        "val_auc"
    ].idxmax()
)

best_frozen_row = (
    frozen_history_df.loc[
        best_frozen_epoch_index
    ]
)

FROZEN_SUMMARY_PATH = (
    METRICS_DIR /
    "frozen_training_summary.json"
)

frozen_summary = {
    "run_id": RUN_ID,
    "stage": "frozen",
    "epochs_completed": int(
        len(frozen_history_df)
    ),
    "maximum_epochs": FROZEN_EPOCHS,
    "trainable_parameters": (
        frozen_trainable_parameters
    ),
    "best_epoch_in_this_stage": int(
        best_frozen_row["epoch"]
    ),
    "best_val_auc_in_this_stage": float(
        best_frozen_row["val_auc"]
    ),
    "best_val_loss_in_this_stage": float(
        frozen_history_df[
            "val_loss"
        ].min()
    ),
    "best_model_path": str(
        BEST_MODEL_PATH
    ),
    "last_model_path": str(
        LAST_MODEL_PATH
    )
}

temporary_summary_path = (
    FROZEN_SUMMARY_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_summary_path,
    "w",
    encoding="utf-8"
) as summary_file:
    json.dump(
        frozen_summary,
        summary_file,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    temporary_summary_path,
    FROZEN_SUMMARY_PATH
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("FROZEN TRAINING COMPLETED")
print("=" * 90)

print(
    "Epochs completed       :",
    len(frozen_history_df)
)

print(
    "Best frozen epoch      :",
    int(best_frozen_row["epoch"])
)

print(
    "Best frozen val AUC    :",
    float(best_frozen_row["val_auc"])
)

print(
    "Minimum frozen val loss:",
    float(
        frozen_history_df[
            "val_loss"
        ].min()
    )
)

print(
    "Best checkpoint        :",
    BEST_MODEL_PATH
)

print(
    "Last checkpoint        :",
    LAST_MODEL_PATH
)

print(
    "History CSV            :",
    FROZEN_HISTORY_PATH
)

print(
    "Summary JSON           :",
    FROZEN_SUMMARY_PATH
)

print("=" * 90)
print("FROZEN TRAINING          : PASS")
print("BEST CHECKPOINT          : VERIFIED")
print("LAST CHECKPOINT          : VERIFIED")
print("RECOVERY BACKUP          : ENABLED")
print("CELL 12 COMPLETED")
print("=" * 90)


FROZEN-BACKBONE TRAINING
Run ID                 : 20260808_0856_mouth_densenet121_seed42
Frozen epochs maximum  : 12
Trainable parameters   : 1,025
Selection metric       : val_auc
Best checkpoint        : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/checkpoints/best.keras
Last checkpoint        : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/checkpoints/last.keras
Recovery backup        : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/checkpoints/training_backup

Starting frozen-backbone training...

Epoch 1/12
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step - accuracy: 0.5234 - auc: 0.5400 - loss: 0.7566 - pr_auc: 0.5385 - precision: 0.5349 - recall: 0.5045
Atomic best checkpoint u

In [15]:
# ============================================================
# CELL 13 — PARTIAL DENSENET121 FINE-TUNING
# ============================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

print("\n" + "=" * 90)
print("PARTIAL DENSENET121 FINE-TUNING")
print("=" * 90)


# ------------------------------------------------------------
# Fine-tuning configuration
# ------------------------------------------------------------
FINETUNE_EPOCHS = 10
FROZEN_EPOCHS_COMPLETED = int(
    frozen_summary["epochs_completed"]
)

FINETUNE_INITIAL_EPOCH = (
    FROZEN_EPOCHS_COMPLETED
)

FINETUNE_FINAL_EPOCH = (
    FINETUNE_INITIAL_EPOCH +
    FINETUNE_EPOCHS
)

FINETUNE_HISTORY_PATH = (
    METRICS_DIR /
    "finetune_training_history.csv"
)

FINETUNE_LOG_PATH = (
    LOG_DIR /
    "finetune_training_log.csv"
)

FINETUNE_SUMMARY_PATH = (
    METRICS_DIR /
    "finetune_training_summary.json"
)

FINETUNE_BACKUP_DIR = (
    CHECKPOINT_DIR /
    "finetune_training_backup"
)


# ------------------------------------------------------------
# Load the best frozen-stage model
# ------------------------------------------------------------
assert BEST_MODEL_PATH.exists(), (
    "The best frozen-stage checkpoint "
    "does not exist."
)

model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

print(
    "Best frozen checkpoint loaded:",
    BEST_MODEL_PATH
)


# ------------------------------------------------------------
# Locate the DenseNet121 backbone
# ------------------------------------------------------------
base_model = model.get_layer(
    "densenet121"
)

# Enable layer-specific fine-tuning
base_model.trainable = True


# ------------------------------------------------------------
# Freeze every backbone layer first
# ------------------------------------------------------------
for layer in base_model.layers:
    layer.trainable = False


# ------------------------------------------------------------
# Unfreeze only the final DenseNet block
#
# DenseNet121 final block:
# conv5_block1 ... conv5_block16
#
# Batch Normalization layers remain frozen.
# ------------------------------------------------------------
unfrozen_layer_names = []
frozen_batch_norm_names = []

for layer in base_model.layers:

    is_final_dense_block = (
        layer.name.startswith(
            "conv5_block"
        )
    )

    is_batch_normalization = isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    )

    if (
        is_final_dense_block
        and not is_batch_normalization
    ):
        layer.trainable = True

        unfrozen_layer_names.append(
            layer.name
        )

    elif (
        is_final_dense_block
        and is_batch_normalization
    ):
        layer.trainable = False

        frozen_batch_norm_names.append(
            layer.name
        )


# ------------------------------------------------------------
# Layer-level quality assertions
# ------------------------------------------------------------
assert unfrozen_layer_names, (
    "No DenseNet121 final-block layers "
    "were unfrozen."
)

assert all(
    not layer.trainable
    for layer in base_model.layers
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    )
), (
    "A Batch Normalization layer was "
    "incorrectly unfrozen."
)

assert all(
    not layer.trainable
    for layer in base_model.layers
    if (
        not layer.name.startswith(
            "conv5_block"
        )
        and not isinstance(
            layer,
            tf.keras.layers.BatchNormalization
        )
    )
), (
    "A layer outside the final dense block "
    "was incorrectly unfrozen."
)


# ------------------------------------------------------------
# Compile with a low fine-tuning learning rate
# ------------------------------------------------------------
finetune_optimizer = (
    tf.keras.optimizers.Adam(
        learning_rate=MODEL_CONFIG[
            "finetune_learning_rate"
        ]
    )
)

model.compile(
    optimizer=finetune_optimizer,

    loss=tf.keras.losses.BinaryCrossentropy(
        name="binary_crossentropy"
    ),

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy",
            threshold=0.5
        ),

        tf.keras.metrics.Precision(
            name="precision",
            thresholds=0.5
        ),

        tf.keras.metrics.Recall(
            name="recall",
            thresholds=0.5
        ),

        tf.keras.metrics.AUC(
            name="auc",
            curve="ROC"
        ),

        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)


# ------------------------------------------------------------
# Parameter accounting
# ------------------------------------------------------------
finetune_total_parameters = (
    model.count_params()
)

finetune_trainable_parameters = int(
    sum(
        tf.keras.backend.count_params(
            variable
        )
        for variable in model.trainable_weights
    )
)

finetune_non_trainable_parameters = (
    finetune_total_parameters -
    finetune_trainable_parameters
)

assert finetune_trainable_parameters > 1025, (
    "Fine-tuning did not add trainable "
    "backbone parameters."
)

assert finetune_trainable_parameters < (
    finetune_total_parameters
), (
    "The complete DenseNet121 model was "
    "accidentally unfrozen."
)


# ------------------------------------------------------------
# Forward-pass numerical quality check
# ------------------------------------------------------------
finetune_test_predictions = model(
    sample_images[:2],
    training=False
).numpy()

assert np.isfinite(
    finetune_test_predictions
).all(), (
    "Fine-tuning model produced NaN or Inf."
)


# ------------------------------------------------------------
# Fine-tuning callbacks
# ------------------------------------------------------------
finetune_atomic_checkpoint = (
    AtomicModelCheckpoint(
        best_model_path=BEST_MODEL_PATH,
        last_model_path=LAST_MODEL_PATH,
        state_path=CHECKPOINT_STATE_PATH,
        monitor="val_auc",
        mode="max"
    )
)

finetune_callbacks = [
    finetune_atomic_checkpoint,

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=str(
            FINETUNE_BACKUP_DIR
        ),
        save_freq="epoch",
        delete_checkpoint=False
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(
            FINETUNE_LOG_PATH
        ),
        separator=",",
        append=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        min_delta=1e-4,
        restore_best_weights=False,
        verbose=1
    ),

    tf.keras.callbacks.TerminateOnNaN(),

    FiniteMetricsGuard()
]


# ------------------------------------------------------------
# Print fine-tuning setup
# ------------------------------------------------------------
print("\n" + "-" * 90)
print("FINE-TUNING CONFIGURATION")
print("-" * 90)

print(
    "Initial checkpoint       :",
    BEST_MODEL_PATH
)

print(
    "Fine-tuning learning rate:",
    MODEL_CONFIG[
        "finetune_learning_rate"
    ]
)

print(
    "Maximum fine-tune epochs :",
    FINETUNE_EPOCHS
)

print(
    "Epoch numbering          :",
    f"{FINETUNE_INITIAL_EPOCH + 1} "
    f"to {FINETUNE_FINAL_EPOCH}"
)

print(
    "Unfrozen backbone layers :",
    len(unfrozen_layer_names)
)

print(
    "Frozen BatchNorm layers  :",
    len(frozen_batch_norm_names)
)

print(
    "Total parameters         :",
    f"{finetune_total_parameters:,}"
)

print(
    "Trainable parameters     :",
    f"{finetune_trainable_parameters:,}"
)

print(
    "Non-trainable parameters :",
    f"{finetune_non_trainable_parameters:,}"
)

print("\nFirst five unfrozen layers:")

for layer_name in unfrozen_layer_names[:5]:
    print(" -", layer_name)

print("\nLast five unfrozen layers:")

for layer_name in unfrozen_layer_names[-5:]:
    print(" -", layer_name)


# ------------------------------------------------------------
# Start partial fine-tuning
# ------------------------------------------------------------
print(
    "\nStarting partial "
    "fine-tuning...\n"
)

finetune_history = model.fit(
    train_dataset,
    validation_data=val_dataset,

    initial_epoch=(
        FINETUNE_INITIAL_EPOCH
    ),

    epochs=FINETUNE_FINAL_EPOCH,

    callbacks=finetune_callbacks,
    verbose=1
)


# ------------------------------------------------------------
# Convert fine-tuning history to a table
# ------------------------------------------------------------
finetune_history_df = pd.DataFrame(
    finetune_history.history
)

finetune_history_df.insert(
    0,
    "epoch",
    np.arange(
        FINETUNE_INITIAL_EPOCH + 1,
        FINETUNE_INITIAL_EPOCH + 1 +
        len(finetune_history_df)
    )
)

finetune_history_df.insert(
    1,
    "stage",
    "finetune"
)


# ------------------------------------------------------------
# Save fine-tuning history atomically
# ------------------------------------------------------------
temporary_history_path = (
    FINETUNE_HISTORY_PATH.with_suffix(
        ".csv.tmp"
    )
)

finetune_history_df.to_csv(
    temporary_history_path,
    index=False
)

os.replace(
    temporary_history_path,
    FINETUNE_HISTORY_PATH
)


# ------------------------------------------------------------
# Combine frozen and fine-tuning histories
# ------------------------------------------------------------
COMBINED_HISTORY_PATH = (
    METRICS_DIR /
    "combined_training_history.csv"
)

combined_history_df = pd.concat(
    [
        frozen_history_df,
        finetune_history_df
    ],
    ignore_index=True,
    sort=False
)

temporary_combined_path = (
    COMBINED_HISTORY_PATH.with_suffix(
        ".csv.tmp"
    )
)

combined_history_df.to_csv(
    temporary_combined_path,
    index=False
)

os.replace(
    temporary_combined_path,
    COMBINED_HISTORY_PATH
)


# ------------------------------------------------------------
# Verify the global best model
# ------------------------------------------------------------
assert BEST_MODEL_PATH.exists(), (
    "Global best checkpoint is missing."
)

assert LAST_MODEL_PATH.exists(), (
    "Fine-tuning last checkpoint is missing."
)

best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

best_model_predictions = best_model(
    sample_images[:2],
    training=False
).numpy()

assert np.isfinite(
    best_model_predictions
).all(), (
    "Global best model produced NaN or Inf."
)


# ------------------------------------------------------------
# Fine-tuning summary
# ------------------------------------------------------------
best_finetune_epoch_index = int(
    finetune_history_df[
        "val_auc"
    ].idxmax()
)

best_finetune_row = (
    finetune_history_df.loc[
        best_finetune_epoch_index
    ]
)

global_best_history_index = int(
    combined_history_df[
        "val_auc"
    ].idxmax()
)

global_best_history_row = (
    combined_history_df.loc[
        global_best_history_index
    ]
)

finetune_summary = {
    "run_id": RUN_ID,
    "stage": "finetune",
    "epochs_completed": int(
        len(finetune_history_df)
    ),
    "maximum_epochs": FINETUNE_EPOCHS,
    "initial_epoch": (
        FINETUNE_INITIAL_EPOCH
    ),
    "final_epoch_number": int(
        finetune_history_df[
            "epoch"
        ].max()
    ),
    "unfrozen_layer_count": len(
        unfrozen_layer_names
    ),
    "frozen_batch_norm_count": len(
        frozen_batch_norm_names
    ),
    "trainable_parameters": (
        finetune_trainable_parameters
    ),
    "best_finetune_epoch": int(
        best_finetune_row["epoch"]
    ),
    "best_finetune_val_auc": float(
        best_finetune_row["val_auc"]
    ),
    "global_best_epoch": int(
        global_best_history_row["epoch"]
    ),
    "global_best_stage": str(
        global_best_history_row["stage"]
    ),
    "global_best_val_auc": float(
        global_best_history_row["val_auc"]
    ),
    "best_model_path": str(
        BEST_MODEL_PATH
    ),
    "last_model_path": str(
        LAST_MODEL_PATH
    )
}

temporary_summary_path = (
    FINETUNE_SUMMARY_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_summary_path,
    "w",
    encoding="utf-8"
) as summary_file:
    json.dump(
        finetune_summary,
        summary_file,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    temporary_summary_path,
    FINETUNE_SUMMARY_PATH
)


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("FINE-TUNING COMPLETED")
print("=" * 90)

print(
    "Fine-tune epochs completed:",
    len(finetune_history_df)
)

print(
    "Best fine-tune epoch     :",
    int(best_finetune_row["epoch"])
)

print(
    "Best fine-tune val AUC   :",
    float(best_finetune_row["val_auc"])
)

print(
    "Global best epoch        :",
    int(global_best_history_row["epoch"])
)

print(
    "Global best stage        :",
    global_best_history_row["stage"]
)

print(
    "Global best val AUC      :",
    float(global_best_history_row["val_auc"])
)

print(
    "Best checkpoint          :",
    BEST_MODEL_PATH
)

print(
    "Combined history         :",
    COMBINED_HISTORY_PATH
)

print(
    "Fine-tune summary        :",
    FINETUNE_SUMMARY_PATH
)

print("=" * 90)
print("PARTIAL FINE-TUNING       : PASS")
print("BATCH NORMALIZATION       : FROZEN")
print("GLOBAL BEST CHECKPOINT    : VERIFIED")
print("COMBINED HISTORY          : SAVED")
print("CELL 13 COMPLETED")
print("=" * 90)


PARTIAL DENSENET121 FINE-TUNING
Best frozen checkpoint loaded: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/checkpoints/best.keras

------------------------------------------------------------------------------------------
FINE-TUNING CONFIGURATION
------------------------------------------------------------------------------------------
Initial checkpoint       : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/checkpoints/best.keras
Fine-tuning learning rate: 1e-05
Maximum fine-tune epochs : 10
Epoch numbering          : 13 to 22
Unfrozen backbone layers : 80
Frozen BatchNorm layers  : 32
Total parameters         : 7,038,529
Trainable parameters     : 2,130,945
Non-trainable parameters : 4,907,584

First five unfrozen layers:
 - conv5_block1_0_relu
 - conv5_block1_1_conv
 - conv5_

In [16]:
# ============================================================
# CELL 14 — FINAL VALIDATION THRESHOLD AND TEST EVALUATION
# ============================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    matthews_corrcoef
)

print("\n" + "=" * 95)
print("FINAL VALIDATION THRESHOLD AND INDEPENDENT TEST EVALUATION")
print("=" * 95)


# ------------------------------------------------------------
# Load the global best checkpoint
# ------------------------------------------------------------
assert BEST_MODEL_PATH.exists(), (
    "Global best checkpoint is missing."
)

best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

print("Global best model:", BEST_MODEL_PATH)
print(
    "Global best validation AUC:",
    finetune_summary["global_best_val_auc"]
)


# ------------------------------------------------------------
# Prediction helper
# ------------------------------------------------------------
def predict_dataset(model_object, dataset):
    labels = []
    probabilities = []

    for image_batch, label_batch in dataset:

        batch_probabilities = (
            model_object(
                image_batch,
                training=False
            )
            .numpy()
            .reshape(-1)
        )

        probabilities.extend(
            batch_probabilities.tolist()
        )

        labels.extend(
            label_batch
            .numpy()
            .astype(int)
            .reshape(-1)
            .tolist()
        )

    labels = np.asarray(
        labels,
        dtype=np.int32
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    assert np.isfinite(
        probabilities
    ).all(), (
        "NaN or Inf detected in predictions."
    )

    assert (
        (probabilities >= 0.0) &
        (probabilities <= 1.0)
    ).all(), (
        "Prediction outside [0, 1]."
    )

    return labels, probabilities


# ------------------------------------------------------------
# Validation predictions
# Validation is used for threshold selection.
# ------------------------------------------------------------
print("\nGenerating validation predictions...")

validation_labels, validation_probabilities = (
    predict_dataset(
        best_model,
        val_dataset
    )
)

assert len(validation_labels) == 296


# ------------------------------------------------------------
# Validation-only Youden J threshold selection
# ------------------------------------------------------------
validation_fpr, validation_tpr, validation_thresholds = (
    roc_curve(
        validation_labels,
        validation_probabilities
    )
)

finite_threshold_mask = np.isfinite(
    validation_thresholds
)

finite_fpr = validation_fpr[
    finite_threshold_mask
]

finite_tpr = validation_tpr[
    finite_threshold_mask
]

finite_thresholds = validation_thresholds[
    finite_threshold_mask
]

youden_j_values = (
    finite_tpr -
    finite_fpr
)

best_threshold_index = int(
    np.argmax(
        youden_j_values
    )
)

SELECTED_THRESHOLD = float(
    finite_thresholds[
        best_threshold_index
    ]
)

SELECTED_YOUDEN_J = float(
    youden_j_values[
        best_threshold_index
    ]
)

validation_predictions = (
    validation_probabilities >=
    SELECTED_THRESHOLD
).astype(int)

validation_auc = roc_auc_score(
    validation_labels,
    validation_probabilities
)

validation_f1 = f1_score(
    validation_labels,
    validation_predictions,
    zero_division=0
)

print("\nValidation-only threshold selection:")
print("Selected threshold :", SELECTED_THRESHOLD)
print("Youden J           :", SELECTED_YOUDEN_J)
print("Validation ROC-AUC :", validation_auc)
print("Validation F1      :", validation_f1)


# ------------------------------------------------------------
# Independent test predictions
# Test is evaluated only after threshold selection.
# ------------------------------------------------------------
print("\nGenerating independent test predictions...")

test_labels, test_probabilities = (
    predict_dataset(
        best_model,
        test_dataset
    )
)

assert len(test_labels) == 302

test_predictions = (
    test_probabilities >=
    SELECTED_THRESHOLD
).astype(int)

default_test_predictions = (
    test_probabilities >= 0.5
).astype(int)


# ------------------------------------------------------------
# Metric calculation function
# ------------------------------------------------------------
def calculate_binary_metrics(
    labels,
    probabilities,
    predictions,
    threshold
):
    matrix = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    )

    true_negative = int(matrix[0, 0])
    false_positive = int(matrix[0, 1])
    false_negative = int(matrix[1, 0])
    true_positive = int(matrix[1, 1])

    specificity_denominator = (
        true_negative +
        false_positive
    )

    specificity = (
        true_negative /
        specificity_denominator
        if specificity_denominator > 0
        else 0.0
    )

    return {
        "threshold": float(threshold),
        "sample_count": int(len(labels)),
        "accuracy": float(
            accuracy_score(
                labels,
                predictions
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                predictions
            )
        ),
        "precision": float(
            precision_score(
                labels,
                predictions,
                zero_division=0
            )
        ),
        "recall": float(
            recall_score(
                labels,
                predictions,
                zero_division=0
            )
        ),
        "specificity": float(
            specificity
        ),
        "f1_score": float(
            f1_score(
                labels,
                predictions,
                zero_division=0
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels,
                probabilities
            )
        ),
        "pr_auc": float(
            average_precision_score(
                labels,
                probabilities
            )
        ),
        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                labels,
                predictions
            )
        ),
        "true_negative": true_negative,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_positive": true_positive
    }


# ------------------------------------------------------------
# Image-level metrics
# ------------------------------------------------------------
image_level_metrics = (
    calculate_binary_metrics(
        labels=test_labels,
        probabilities=test_probabilities,
        predictions=test_predictions,
        threshold=SELECTED_THRESHOLD
    )
)

default_threshold_metrics = (
    calculate_binary_metrics(
        labels=test_labels,
        probabilities=test_probabilities,
        predictions=default_test_predictions,
        threshold=0.5
    )
)


# ------------------------------------------------------------
# Align test predictions with the standard manifest
# ------------------------------------------------------------
test_manifest = (
    model_manifest.loc[
        model_manifest["split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)

expected_test_paths = [
    str(path)
    for path in dataset_records[
        "test"
    ]["paths"]
]

assert test_manifest[
    "output_path"
].tolist() == expected_test_paths, (
    "Test prediction order does not match "
    "the standard manifest."
)

assert np.array_equal(
    test_manifest["label"]
    .map(LABEL_MAP)
    .to_numpy(dtype=int),
    test_labels
), (
    "Test labels do not match manifest labels."
)

test_predictions_df = test_manifest[
    [
        "sample_id",
        "source_video",
        "frame_index",
        "face_index",
        "label",
        "sha256",
        "output_path"
    ]
].copy()

test_predictions_df[
    "true_label"
] = test_labels

test_predictions_df[
    "fake_probability"
] = test_probabilities

test_predictions_df[
    "selected_threshold"
] = SELECTED_THRESHOLD

test_predictions_df[
    "predicted_label"
] = test_predictions

test_predictions_df[
    "predicted_class"
] = np.where(
    test_predictions == 1,
    "fake",
    "real"
)

test_predictions_df[
    "is_correct"
] = (
    test_predictions_df[
        "true_label"
    ] ==
    test_predictions_df[
        "predicted_label"
    ]
)


# ------------------------------------------------------------
# Source/video-level aggregation
# Multiple face ROIs belonging to the same source are averaged.
# ------------------------------------------------------------
label_consistency = (
    test_predictions_df
    .groupby("source_video")[
        "true_label"
    ]
    .nunique()
)

assert (
    label_consistency == 1
).all(), (
    "A source_video has conflicting labels."
)

source_predictions_df = (
    test_predictions_df
    .groupby(
        "source_video",
        as_index=False
    )
    .agg(
        true_label=(
            "true_label",
            "first"
        ),
        fake_probability=(
            "fake_probability",
            "mean"
        ),
        roi_count=(
            "sample_id",
            "count"
        )
    )
)

source_predictions_df[
    "selected_threshold"
] = SELECTED_THRESHOLD

source_predictions_df[
    "predicted_label"
] = (
    source_predictions_df[
        "fake_probability"
    ] >= SELECTED_THRESHOLD
).astype(int)

source_predictions_df[
    "predicted_class"
] = np.where(
    source_predictions_df[
        "predicted_label"
    ] == 1,
    "fake",
    "real"
)

source_predictions_df[
    "is_correct"
] = (
    source_predictions_df[
        "true_label"
    ] ==
    source_predictions_df[
        "predicted_label"
    ]
)

source_level_metrics = (
    calculate_binary_metrics(
        labels=source_predictions_df[
            "true_label"
        ].to_numpy(dtype=int),

        probabilities=source_predictions_df[
            "fake_probability"
        ].to_numpy(dtype=float),

        predictions=source_predictions_df[
            "predicted_label"
        ].to_numpy(dtype=int),

        threshold=SELECTED_THRESHOLD
    )
)


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------
classification_report_dict = (
    classification_report(
        test_labels,
        test_predictions,
        labels=[0, 1],
        target_names=[
            "Real",
            "Fake"
        ],
        output_dict=True,
        zero_division=0
    )
)

classification_report_df = (
    pd.DataFrame(
        classification_report_dict
    )
    .transpose()
)


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------
IMAGE_METRICS_PATH = (
    METRICS_DIR /
    "final_image_level_metrics.csv"
)

SOURCE_METRICS_PATH = (
    METRICS_DIR /
    "final_source_level_metrics.csv"
)

DEFAULT_METRICS_PATH = (
    METRICS_DIR /
    "default_threshold_metrics.csv"
)

CLASSIFICATION_REPORT_PATH = (
    METRICS_DIR /
    "classification_report.csv"
)

THRESHOLD_PATH = (
    METRICS_DIR /
    "validation_threshold.json"
)

TEST_PREDICTIONS_PATH = (
    PREDICTION_DIR /
    "test_image_predictions.csv"
)

SOURCE_PREDICTIONS_PATH = (
    PREDICTION_DIR /
    "test_source_predictions.csv"
)


# ------------------------------------------------------------
# Atomic table writer
# ------------------------------------------------------------
def atomic_write_csv(
    dataframe,
    output_path
):
    output_path = Path(output_path)

    temporary_path = (
        output_path.with_suffix(
            ".csv.tmp"
        )
    )

    dataframe.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        output_path
    )


atomic_write_csv(
    pd.DataFrame(
        [image_level_metrics]
    ),
    IMAGE_METRICS_PATH
)

atomic_write_csv(
    pd.DataFrame(
        [source_level_metrics]
    ),
    SOURCE_METRICS_PATH
)

atomic_write_csv(
    pd.DataFrame(
        [default_threshold_metrics]
    ),
    DEFAULT_METRICS_PATH
)

atomic_write_csv(
    classification_report_df.reset_index(
        names="class"
    ),
    CLASSIFICATION_REPORT_PATH
)

atomic_write_csv(
    test_predictions_df,
    TEST_PREDICTIONS_PATH
)

atomic_write_csv(
    source_predictions_df,
    SOURCE_PREDICTIONS_PATH
)


# ------------------------------------------------------------
# Save validation threshold atomically
# ------------------------------------------------------------
threshold_data = {
    "run_id": RUN_ID,
    "selection_dataset": "validation",
    "selection_method": "Youden J",
    "selected_threshold": SELECTED_THRESHOLD,
    "youden_j": SELECTED_YOUDEN_J,
    "validation_roc_auc": float(
        validation_auc
    ),
    "validation_f1_at_selected_threshold": float(
        validation_f1
    ),
    "test_set_used_for_threshold_selection": False
}

temporary_threshold_path = (
    THRESHOLD_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_threshold_path,
    "w",
    encoding="utf-8"
) as threshold_file:
    json.dump(
        threshold_data,
        threshold_file,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    temporary_threshold_path,
    THRESHOLD_PATH
)


# ------------------------------------------------------------
# Plot configuration
# All report figures use English labels.
# ------------------------------------------------------------
sns.set_theme(
    style="whitegrid",
    context="notebook"
)

TRAIN_COLOR = "#2878B5"
VALIDATION_COLOR = "#E07A2D"
REAL_COLOR = "#2A9D8F"
FAKE_COLOR = "#D1495B"


def save_figure_with_quality_gate(
    figure,
    output_path
):
    output_path = Path(output_path)

    temporary_path = (
        output_path.parent /
        f"{output_path.stem}.tmp.png"
    )

    figure.savefig(
        temporary_path,
        dpi=150,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(figure)

    with Image.open(
        temporary_path
    ) as image_object:
        image_size = image_object.size

    assert min(image_size) >= 600, (
        f"Figure resolution is too low: "
        f"{image_size}"
    )

    os.replace(
        temporary_path,
        output_path
    )

    return image_size


# ------------------------------------------------------------
# Figure 1 — Combined training curves
# ------------------------------------------------------------
TRAINING_CURVES_PATH = (
    FIGURE_DIR /
    "training_validation_curves.png"
)

figure, axes = plt.subplots(
    3,
    1,
    figsize=(11, 14),
    dpi=150,
    sharex=True
)

epoch_values = combined_history_df[
    "epoch"
].to_numpy()

axes[0].plot(
    epoch_values,
    combined_history_df["loss"],
    label="Training Loss",
    color=TRAIN_COLOR,
    linewidth=2
)

axes[0].plot(
    epoch_values,
    combined_history_df["val_loss"],
    label="Validation Loss",
    color=VALIDATION_COLOR,
    linewidth=2,
    linestyle="--"
)

axes[0].set_title(
    "Training and Validation Loss",
    fontsize=14,
    fontweight="bold"
)

axes[0].set_ylabel(
    "Binary Cross-Entropy Loss",
    fontsize=11
)

axes[1].plot(
    epoch_values,
    combined_history_df["auc"],
    label="Training ROC-AUC",
    color=TRAIN_COLOR,
    linewidth=2
)

axes[1].plot(
    epoch_values,
    combined_history_df["val_auc"],
    label="Validation ROC-AUC",
    color=VALIDATION_COLOR,
    linewidth=2,
    linestyle="--"
)

axes[1].set_title(
    "Training and Validation ROC-AUC",
    fontsize=14,
    fontweight="bold"
)

axes[1].set_ylabel(
    "ROC-AUC",
    fontsize=11
)

axes[2].plot(
    epoch_values,
    combined_history_df["accuracy"],
    label="Training Accuracy",
    color=TRAIN_COLOR,
    linewidth=2
)

axes[2].plot(
    epoch_values,
    combined_history_df["val_accuracy"],
    label="Validation Accuracy",
    color=VALIDATION_COLOR,
    linewidth=2,
    linestyle="--"
)

axes[2].set_title(
    "Training and Validation Accuracy",
    fontsize=14,
    fontweight="bold"
)

axes[2].set_xlabel(
    "Epoch",
    fontsize=11
)

axes[2].set_ylabel(
    "Accuracy",
    fontsize=11
)

for axis in axes:
    axis.axvline(
        FROZEN_EPOCHS_COMPLETED + 0.5,
        color="#6C757D",
        linestyle=":",
        linewidth=2,
        label="Fine-Tuning Start"
    )

    axis.legend(
        frameon=True,
        fontsize=10
    )

    axis.grid(
        True,
        alpha=0.25
    )

figure.tight_layout()

training_curve_size = (
    save_figure_with_quality_gate(
        figure,
        TRAINING_CURVES_PATH
    )
)


# ------------------------------------------------------------
# Figure 2 — Image and source confusion matrices
# ------------------------------------------------------------
CONFUSION_MATRIX_PATH = (
    FIGURE_DIR /
    "test_confusion_matrices_matrices.png"
)

image_confusion = confusion_matrix(
    test_labels,
    test_predictions,
    labels=[0, 1]
)

source_confusion = confusion_matrix(
    source_predictions_df["true_label"],
    source_predictions_df["predicted_label"],
    labels=[0, 1]
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5.5),
    dpi=150
)

sns.heatmap(
    image_confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    square=True,
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"],
    ax=axes[0]
)

axes[0].set_title(
    "Image-Level Test Confusion Matrix",
    fontsize=13,
    fontweight="bold"
)

axes[0].set_xlabel(
    "Predicted Label",
    fontsize=11
)

axes[0].set_ylabel(
    "True Label",
    fontsize=11
)

sns.heatmap(
    source_confusion,
    annot=True,
    fmt="d",
    cmap="Greens",
    cbar=False,
    square=True,
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"],
    ax=axes[1]
)

axes[1].set_title(
    "Source-Level Test Confusion Matrix",
    fontsize=13,
    fontweight="bold"
)

axes[1].set_xlabel(
    "Predicted Label",
    fontsize=11
)

axes[1].set_ylabel(
    "True Label",
    fontsize=11
)

figure.tight_layout()

confusion_matrix_size = (
    save_figure_with_quality_gate(
        figure,
        CONFUSION_MATRIX_PATH
    )
)


# ------------------------------------------------------------
# Figure 3 — ROC and Precision-Recall curves
# ------------------------------------------------------------
ROC_PR_PATH = (
    FIGURE_DIR /
    "test_roc_pr_curves.png"
)

test_fpr, test_tpr, _ = roc_curve(
    test_labels,
    test_probabilities
)

test_precision_curve, test_recall_curve, _ = (
    precision_recall_curve(
        test_labels,
        test_probabilities
    )
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
    dpi=150
)

axes[0].plot(
    test_fpr,
    test_tpr,
    color=TRAIN_COLOR,
    linewidth=2.5,
    label=(
        f"ROC-AUC = "
        f"{image_level_metrics['roc_auc']:.3f}"
    )
)

axes[0].plot(
    [0, 1],
    [0, 1],
    color="#777777",
    linestyle="--",
    linewidth=1.5,
    label="Random Classifier"
)

axes[0].set_title(
    "Image-Level Test ROC Curve",
    fontsize=13,
    fontweight="bold"
)

axes[0].set_xlabel(
    "False Positive Rate",
    fontsize=11
)

axes[0].set_ylabel(
    "True Positive Rate",
    fontsize=11
)

axes[0].legend(
    frameon=True
)

axes[0].grid(
    True,
    alpha=0.25
)

axes[1].plot(
    test_recall_curve,
    test_precision_curve,
    color=FAKE_COLOR,
    linewidth=2.5,
    label=(
        f"PR-AUC = "
        f"{image_level_metrics['pr_auc']:.3f}"
    )
)

axes[1].set_title(
    "Image-Level Test Precision-Recall Curve",
    fontsize=13,
    fontweight="bold"
)

axes[1].set_xlabel(
    "Recall",
    fontsize=11
)

axes[1].set_ylabel(
    "Precision",
    fontsize=11
)

axes[1].legend(
    frameon=True
)

axes[1].grid(
    True,
    alpha=0.25
)

figure.tight_layout()

roc_pr_size = (
    save_figure_with_quality_gate(
        figure,
        ROC_PR_PATH
    )
)


# ------------------------------------------------------------
# Figure 4 — Probability distribution
# ------------------------------------------------------------
PROBABILITY_DISTRIBUTION_PATH = (
    FIGURE_DIR /
    "test_probability_distribution.png"
)

figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150
)

sns.histplot(
    test_probabilities[
        test_labels == 0
    ],
    bins=20,
    stat="density",
    alpha=0.55,
    color=REAL_COLOR,
    label="Real",
    ax=axis
)

sns.histplot(
    test_probabilities[
        test_labels == 1
    ],
    bins=20,
    stat="density",
    alpha=0.55,
    color=FAKE_COLOR,
    label="Fake",
    ax=axis
)

axis.axvline(
    SELECTED_THRESHOLD,
    color="#222222",
    linestyle="--",
    linewidth=2,
    label=(
        f"Validation Threshold = "
        f"{SELECTED_THRESHOLD:.3f}"
    )
)

axis.set_title(
    "Test Fake-Probability Distribution",
    fontsize=14,
    fontweight="bold"
)

axis.set_xlabel(
    "Predicted Fake Probability",
    fontsize=11
)

axis.set_ylabel(
    "Density",
    fontsize=11
)

axis.legend(
    frameon=True
)

axis.grid(
    True,
    alpha=0.25
)

figure.tight_layout()

probability_size = (
    save_figure_with_quality_gate(
        figure,
        PROBABILITY_DISTRIBUTION_PATH
    )
)


# ------------------------------------------------------------
# Final evaluation summary
# ------------------------------------------------------------
FINAL_EVALUATION_PATH = (
    METRICS_DIR /
    "final_evaluation_summary.json"
)

final_evaluation_summary = {
    "run_id": RUN_ID,
    "model": "DenseNet121",
    "global_best_epoch": int(
        finetune_summary[
            "global_best_epoch"
        ]
    ),
    "global_best_stage": str(
        finetune_summary[
            "global_best_stage"
        ]
    ),
    "global_best_validation_auc": float(
        finetune_summary[
            "global_best_val_auc"
        ]
    ),
    "threshold_selection": threshold_data,
    "image_level_test_metrics": (
        image_level_metrics
    ),
    "source_level_test_metrics": (
        source_level_metrics
    ),
    "default_threshold_test_metrics": (
        default_threshold_metrics
    ),
    "test_used_once_for_final_evaluation": True,
    "figures": {
        "training_curves": {
            "path": str(
                TRAINING_CURVES_PATH
            ),
            "size": training_curve_size
        },
        "confusion_matrices": {
            "path": str(
                CONFUSION_MATRIX_PATH
            ),
            "size": confusion_matrix_size
        },
        "roc_pr_curves": {
            "path": str(
                ROC_PR_PATH
            ),
            "size": roc_pr_size
        },
        "probability_distribution": {
            "path": str(
                PROBABILITY_DISTRIBUTION_PATH
            ),
            "size": probability_size
        }
    }
}

temporary_final_path = (
    FINAL_EVALUATION_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_final_path,
    "w",
    encoding="utf-8"
) as final_file:
    json.dump(
        final_evaluation_summary,
        final_file,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    temporary_final_path,
    FINAL_EVALUATION_PATH
)


# ------------------------------------------------------------
# Print final test results
# ------------------------------------------------------------
print("\n" + "=" * 95)
print("FINAL INDEPENDENT TEST RESULTS")
print("=" * 95)

print(
    "Validation-selected threshold:",
    f"{SELECTED_THRESHOLD:.6f}"
)

print("\nIMAGE-LEVEL TEST METRICS")

for metric_name, metric_value in (
    image_level_metrics.items()
):
    if isinstance(metric_value, float):
        print(
            f"{metric_name:35}: "
            f"{metric_value:.6f}"
        )
    else:
        print(
            f"{metric_name:35}: "
            f"{metric_value}"
        )

print("\nSOURCE-LEVEL TEST METRICS")

for metric_name, metric_value in (
    source_level_metrics.items()
):
    if isinstance(metric_value, float):
        print(
            f"{metric_name:35}: "
            f"{metric_value:.6f}"
        )
    else:
        print(
            f"{metric_name:35}: "
            f"{metric_value}"
        )

print("\nSaved figures:")
print(" -", TRAINING_CURVES_PATH)
print(" -", CONFUSION_MATRIX_PATH)
print(" -", ROC_PR_PATH)
print(" -", PROBABILITY_DISTRIBUTION_PATH)

print("=" * 95)
print("VALIDATION THRESHOLD SELECTION : PASS")
print("INDEPENDENT TEST EVALUATION    : PASS")
print("IMAGE-LEVEL METRICS            : SAVED")
print("SOURCE-LEVEL METRICS           : SAVED")
print("FIGURE QUALITY GATES           : PASS")
print("CELL 14 COMPLETED")
print("=" * 95)


FINAL VALIDATION THRESHOLD AND INDEPENDENT TEST EVALUATION
Global best model: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/checkpoints/best.keras
Global best validation AUC: 0.7799587845802307

Generating validation predictions...

Validation-only threshold selection:
Selected threshold : 0.508567750453949
Youden J           : 0.4562800274536719
Validation ROC-AUC : 0.7795470144131778
Validation F1      : 0.7272727272727273

Generating independent test predictions...

FINAL INDEPENDENT TEST RESULTS
Validation-selected threshold: 0.508568

IMAGE-LEVEL TEST METRICS
threshold                          : 0.508568
sample_count                       : 302
accuracy                           : 0.695364
balanced_accuracy                  : 0.692615
precision                          : 0.679775
recall                             : 0.775641
specificity                        : 0.609589
f1_scor

In [1]:
print(RUN_ID)

NameError: name 'RUN_ID' is not defined

In [2]:
# ============================================================
# RECOVERY CELL — RESTORE COMPLETED DENSENET121 RUN
# No training is performed.
# ============================================================

from google.colab import drive
from pathlib import Path

import json
import unicodedata
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras import mixed_precision


# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")


# ------------------------------------------------------------
# Unicode-safe folder detection
# ------------------------------------------------------------
def normalize_name(name):
    return unicodedata.normalize(
        "NFC",
        str(name)
    ).strip().casefold()


def find_child_folder(
    parent,
    expected_name
):
    parent = Path(parent)

    if not parent.exists():
        raise FileNotFoundError(
            f"Parent folder not found: {parent}"
        )

    normalized_expected = normalize_name(
        expected_name
    )

    for item in parent.iterdir():

        if (
            item.is_dir()
            and normalize_name(item.name) ==
            normalized_expected
        ):
            return item

    available_folders = [
        item.name
        for item in parent.iterdir()
        if item.is_dir()
    ]

    raise FileNotFoundError(
        f"\nFolder not found: {expected_name}\n"
        f"Parent folder: {parent}\n"
        f"Available folders: {available_folders}"
    )


# ------------------------------------------------------------
# Recover exact experiment path
# ------------------------------------------------------------
MY_DRIVE = Path(
    "/content/drive/MyDrive"
)

AISC_ROOT = find_child_folder(
    MY_DRIVE,
    "AISC DeepFake Çalışmaları"
)

EXPERIMENTS_ROOT = find_child_folder(
    AISC_ROOT,
    "Deneyler"
)

DILARA_ROOT = find_child_folder(
    EXPERIMENTS_ROOT,
    "Dilara"
)

EXPERIMENT_ROOT = find_child_folder(
    DILARA_ROOT,
    "Deney 1"
)

RESULTS_ROOT = find_child_folder(
    EXPERIMENT_ROOT,
    "Sonuçlar"
)

MODEL_RESULTS_ROOT = find_child_folder(
    RESULTS_ROOT,
    "DenseNet121_Mouth_Results"
)


# ------------------------------------------------------------
# Restore the completed Run ID
# ------------------------------------------------------------
RUN_ID = (
    "20260808_0856_"
    "mouth_densenet121_seed42"
)

RUN_ROOT = MODEL_RESULTS_ROOT / RUN_ID

assert RUN_ROOT.exists(), (
    f"Completed run folder was not found:\n"
    f"{RUN_ROOT}"
)


# ------------------------------------------------------------
# Restore standard directories
# ------------------------------------------------------------
CHECKPOINT_DIR = (
    RUN_ROOT /
    "checkpoints"
)

LOG_DIR = (
    RUN_ROOT /
    "logs"
)

METRICS_DIR = (
    RUN_ROOT /
    "metrics"
)

PREDICTION_DIR = (
    RUN_ROOT /
    "predictions"
)

FIGURE_DIR = (
    RUN_ROOT /
    "figures"
)

METADATA_DIR = (
    RUN_ROOT /
    "metadata"
)


# ------------------------------------------------------------
# Restore important file paths
# ------------------------------------------------------------
BEST_MODEL_PATH = (
    CHECKPOINT_DIR /
    "best.keras"
)

LAST_MODEL_PATH = (
    CHECKPOINT_DIR /
    "last.keras"
)

MANIFEST_PATH = (
    METADATA_DIR /
    "model_manifest.csv"
)

LEAKAGE_REPORT_PATH = (
    METADATA_DIR /
    "leakage_check.csv"
)

FINETUNE_SUMMARY_PATH = (
    METRICS_DIR /
    "finetune_training_summary.json"
)

THRESHOLD_PATH = (
    METRICS_DIR /
    "validation_threshold.json"
)


# ------------------------------------------------------------
# Verify saved files
# ------------------------------------------------------------
recovery_required_files = {
    "best model": BEST_MODEL_PATH,
    "last model": LAST_MODEL_PATH,
    "manifest": MANIFEST_PATH,
    "leakage report": LEAKAGE_REPORT_PATH,
    "fine-tune summary": FINETUNE_SUMMARY_PATH,
    "validation threshold": THRESHOLD_PATH
}

print("\nRecovered files:")
print("-" * 85)

for file_name, file_path in (
    recovery_required_files.items()
):
    assert file_path.exists(), (
        f"Required recovery file is missing:\n"
        f"{file_path}"
    )

    assert file_path.stat().st_size > 0, (
        f"Recovery file is empty:\n"
        f"{file_path}"
    )

    print(
        f"{file_name:24}: "
        f"FOUND | {file_path}"
    )


# ------------------------------------------------------------
# Restore saved configuration values
# ------------------------------------------------------------
with open(
    FINETUNE_SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as summary_file:
    finetune_summary = json.load(
        summary_file
    )

with open(
    THRESHOLD_PATH,
    "r",
    encoding="utf-8"
) as threshold_file:
    threshold_data = json.load(
        threshold_file
    )

SELECTED_THRESHOLD = float(
    threshold_data[
        "selected_threshold"
    ]
)

LABEL_MAP = {
    "real": 0,
    "fake": 1
}

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_SIZE = (
    IMAGE_HEIGHT,
    IMAGE_WIDTH
)

BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE


# ------------------------------------------------------------
# Restore standard manifest
# ------------------------------------------------------------
model_manifest = pd.read_csv(
    MANIFEST_PATH
)

assert len(model_manifest) == 2987

assert model_manifest[
    "sample_id"
].is_unique

assert model_manifest[
    "output_path"
].notna().all()


# ------------------------------------------------------------
# Reconstruct test dataset from saved manifest
# ------------------------------------------------------------
test_manifest = (
    model_manifest.loc[
        model_manifest["split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(test_manifest) == 302

test_image_paths = (
    test_manifest[
        "output_path"
    ]
    .astype(str)
    .tolist()
)

test_labels = (
    test_manifest[
        "label"
    ]
    .str.lower()
    .map(LABEL_MAP)
    .astype(int)
    .tolist()
)


def recovery_decode_and_preprocess(
    image_path,
    label
):
    image_bytes = tf.io.read_file(
        image_path
    )

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape(
        [None, None, 3]
    )

    image = tf.image.resize(
        image,
        size=IMAGE_SIZE,
        method=tf.image.ResizeMethod.BILINEAR,
        antialias=True
    )

    image = tf.cast(
        image,
        tf.float32
    )

    image = preprocess_input(
        image
    )

    label = tf.cast(
        label,
        tf.float32
    )

    return image, label


test_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        (
            test_image_paths,
            test_labels
        )
    )
    .map(
        recovery_decode_and_preprocess,
        num_parallel_calls=AUTOTUNE,
        deterministic=True
    )
    .batch(
        BATCH_SIZE,
        drop_remainder=False
    )
    .prefetch(
        AUTOTUNE
    )
)


# ------------------------------------------------------------
# Restore mixed-precision policy
# ------------------------------------------------------------
mixed_precision.set_global_policy(
    "mixed_float16"
)


# ------------------------------------------------------------
# Test one batch
# ------------------------------------------------------------
recovery_images, recovery_labels = next(
    iter(test_dataset)
)

assert recovery_images.shape == (
    16,
    224,
    224,
    3
)

assert recovery_labels.shape == (16,)

assert bool(
    tf.reduce_all(
        tf.math.is_finite(
            recovery_images
        )
    )
)


# ------------------------------------------------------------
# Load best model once for recovery verification
# ------------------------------------------------------------
recovery_model = (
    tf.keras.models.load_model(
        BEST_MODEL_PATH
    )
)

recovery_probabilities = (
    recovery_model(
        recovery_images[:2],
        training=False
    )
    .numpy()
    .reshape(-1)
)

assert np.isfinite(
    recovery_probabilities
).all()

assert (
    (recovery_probabilities >= 0.0) &
    (recovery_probabilities <= 1.0)
).all()


# ------------------------------------------------------------
# Recovery report
# ------------------------------------------------------------
print("\n" + "=" * 85)
print("COMPLETED RUN RECOVERED")
print("=" * 85)

print("Run ID              :", RUN_ID)
print("Run root            :", RUN_ROOT)
print("Best model          :", BEST_MODEL_PATH)
print(
    "Global best epoch   :",
    finetune_summary[
        "global_best_epoch"
    ]
)
print(
    "Global best val AUC :",
    finetune_summary[
        "global_best_val_auc"
    ]
)
print(
    "Selected threshold  :",
    SELECTED_THRESHOLD
)
print(
    "Manifest rows       :",
    len(model_manifest)
)
print(
    "Test rows           :",
    len(test_manifest)
)
print(
    "Recovery predictions:",
    recovery_probabilities.tolist()
)

print("=" * 85)
print("TRAINING RESULTS     : FOUND")
print("BEST MODEL           : VERIFIED")
print("TEST DATASET         : RESTORED")
print("RECOVERY STATUS      : PASS")
print("=" * 85)

Mounted at /content/drive

Recovered files:
-------------------------------------------------------------------------------------
best model              : FOUND | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/checkpoints/best.keras
last model              : FOUND | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/checkpoints/last.keras
manifest                : FOUND | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/metadata/model_manifest.csv
leakage report          : FOUND | /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/metadata/leakage_check.csv
fine-tune summary       : FOUND |

In [6]:
# ============================================================
# CELL 15 — FINAL AUDIT AND ZIP EXPORT
# Run this after the Recovery Cell.
# ============================================================

import os
import json
import shutil
import hashlib
import zipfile
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn
import PIL
import yaml
import tensorflow as tf

from PIL import Image


print("\n" + "=" * 100)
print("FINAL AUDIT, CLEAN-LOAD INFERENCE TEST AND ZIP EXPORT")
print("=" * 100)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def atomic_write_text(output_path, content):
    output_path = Path(output_path)

    temporary_path = (
        output_path.parent /
        f"{output_path.name}.tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8"
    ) as output_file:
        output_file.write(content)

    os.replace(
        temporary_path,
        output_path
    )


def atomic_write_json(output_path, content):
    output_path = Path(output_path)

    temporary_path = (
        output_path.parent /
        f"{output_path.name}.tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8"
    ) as output_file:
        json.dump(
            content,
            output_file,
            indent=4,
            ensure_ascii=False
        )

    os.replace(
        temporary_path,
        output_path
    )


def atomic_write_csv(dataframe, output_path):
    output_path = Path(output_path)

    temporary_path = (
        output_path.parent /
        f"{output_path.name}.tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        output_path
    )


def calculate_sha256(file_path):
    sha256_object = hashlib.sha256()

    with open(file_path, "rb") as file_stream:

        while True:
            file_chunk = file_stream.read(
                1024 * 1024
            )

            if not file_chunk:
                break

            sha256_object.update(
                file_chunk
            )

    return sha256_object.hexdigest()


# ------------------------------------------------------------
# Required directories
# ------------------------------------------------------------
required_directories = {
    "checkpoints": CHECKPOINT_DIR,
    "logs": LOG_DIR,
    "metrics": METRICS_DIR,
    "predictions": PREDICTION_DIR,
    "figures": FIGURE_DIR,
    "metadata": METADATA_DIR
}

print("\nRequired directories:")
print("-" * 100)

for directory_name, directory_path in (
    required_directories.items()
):
    assert directory_path.exists(), (
        f"Missing directory: {directory_path}"
    )

    print(
        f"{directory_name:15}: FOUND"
    )


# ------------------------------------------------------------
# Required final files
# ------------------------------------------------------------
required_files = {
    "resolved_config": (
        RUN_ROOT /
        "config_resolved.yaml"
    ),

    "best_model": BEST_MODEL_PATH,

    "last_model": LAST_MODEL_PATH,

    "model_manifest": MANIFEST_PATH,

    "leakage_report": LEAKAGE_REPORT_PATH,

    "accounting_report": (
        METADATA_DIR /
        "accounting_summary.json"
    ),

    "smoke_test_report": (
        METRICS_DIR /
        "smoke_test_report.json"
    ),

    "frozen_history": (
        METRICS_DIR /
        "frozen_training_history.csv"
    ),

    "finetune_history": (
        METRICS_DIR /
        "finetune_training_history.csv"
    ),

    "combined_history": (
        METRICS_DIR /
        "combined_training_history.csv"
    ),

    "image_metrics": (
        METRICS_DIR /
        "final_image_level_metrics.csv"
    ),

    "source_metrics": (
        METRICS_DIR /
        "final_source_level_metrics.csv"
    ),

    "classification_report": (
        METRICS_DIR /
        "classification_report.csv"
    ),

    "validation_threshold": (
        METRICS_DIR /
        "validation_threshold.json"
    ),

    "final_evaluation": (
        METRICS_DIR /
        "final_evaluation_summary.json"
    ),

    "image_predictions": (
        PREDICTION_DIR /
        "test_image_predictions.csv"
    ),

    "source_predictions": (
        PREDICTION_DIR /
        "test_source_predictions.csv"
    ),

    "training_curves": (
        FIGURE_DIR /
        "training_validation_curves.png"
    ),

    "confusion_matrices": (
        FIGURE_DIR /
        "test_confusion_matrices.png"
    ),

    "roc_pr_curves": (
        FIGURE_DIR /
        "test_roc_pr_curves.png"
    ),

    "probability_distribution": (
        FIGURE_DIR /
        "test_probability_distribution.png"
    )
}


print("\nRequired files:")
print("-" * 100)

for file_name, file_path in required_files.items():

    assert file_path.exists(), (
        f"Missing file: {file_path}"
    )

    assert file_path.stat().st_size > 0, (
        f"Empty file: {file_path}"
    )

    print(
        f"{file_name:28}: FOUND"
    )


# ------------------------------------------------------------
# Reload final tables
# ------------------------------------------------------------
saved_manifest = pd.read_csv(
    MANIFEST_PATH
)

saved_leakage_report = pd.read_csv(
    LEAKAGE_REPORT_PATH
)

saved_image_metrics = pd.read_csv(
    required_files["image_metrics"]
)

saved_source_metrics = pd.read_csv(
    required_files["source_metrics"]
)

saved_image_predictions = pd.read_csv(
    required_files["image_predictions"]
)

saved_source_predictions = pd.read_csv(
    required_files["source_predictions"]
)

saved_combined_history = pd.read_csv(
    required_files["combined_history"]
)


# ------------------------------------------------------------
# Table quality gates
# ------------------------------------------------------------
assert len(saved_manifest) == 2987
assert saved_manifest["sample_id"].is_unique
assert saved_manifest["sha256"].notna().all()
assert saved_manifest["output_path"].notna().all()

assert len(saved_image_predictions) == 302
assert len(saved_source_predictions) == 292
assert len(saved_combined_history) == 22

assert (
    saved_leakage_report[
        "source_video_overlap"
    ] == 0
).all()

assert (
    saved_leakage_report[
        "sha256_overlap"
    ] == 0
).all()

assert np.isfinite(
    saved_image_metrics.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

assert np.isfinite(
    saved_source_metrics.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

print("\nTable quality gates: PASS")


# ------------------------------------------------------------
# Clean-load inference test
# ------------------------------------------------------------
print("\nRunning clean-load inference test...")

fresh_model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

fresh_images, fresh_labels = next(
    iter(test_dataset)
)

fresh_probabilities = (
    fresh_model(
        fresh_images[:4],
        training=False
    )
    .numpy()
    .reshape(-1)
)

assert fresh_probabilities.shape == (4,)

assert np.isfinite(
    fresh_probabilities
).all()

assert (
    (fresh_probabilities >= 0.0) &
    (fresh_probabilities <= 1.0)
).all()

print(
    "Clean-load probabilities:",
    fresh_probabilities.tolist()
)

print("Clean-load inference: PASS")


# ------------------------------------------------------------
# Figure quality gates
# ------------------------------------------------------------
figure_quality_records = []

figure_keys = [
    "training_curves",
    "confusion_matrices",
    "roc_pr_curves",
    "probability_distribution"
]

print("\nFigure quality checks:")
print("-" * 100)

for figure_name in figure_keys:

    figure_path = required_files[
        figure_name
    ]

    with Image.open(
        figure_path
    ) as figure_image:
        width, height = figure_image.size

    minimum_side = min(
        width,
        height
    )

    assert minimum_side >= 600, (
        f"Low-resolution figure: "
        f"{figure_path} = "
        f"{width} x {height}"
    )

    figure_quality_records.append(
        {
            "figure": figure_name,
            "path": str(figure_path),
            "width_px": int(width),
            "height_px": int(height),
            "minimum_side_px": int(
                minimum_side
            ),
            "status": "PASS"
        }
    )

    print(
        f"{figure_name:28}: "
        f"{width} x {height} px | PASS"
    )


FIGURE_AUDIT_PATH = (
    METRICS_DIR /
    "figure_quality_audit.csv"
)

atomic_write_csv(
    pd.DataFrame(
        figure_quality_records
    ),
    FIGURE_AUDIT_PATH
)


# ------------------------------------------------------------
# Runtime dependency snapshot
# ------------------------------------------------------------
REQUIREMENTS_PATH = (
    RUN_ROOT /
    "requirements_snapshot.txt"
)

requirements_text = "\n".join(
    [
        f"python=={platform.python_version()}",
        f"tensorflow=={tf.__version__}",
        f"numpy=={np.__version__}",
        f"pandas=={pd.__version__}",
        f"scikit-learn=={sklearn.__version__}",
        f"matplotlib=={matplotlib.__version__}",
        f"seaborn=={seaborn.__version__}",
        f"pillow=={PIL.__version__}",
        f"pyyaml=={yaml.__version__}"
    ]
) + "\n"

atomic_write_text(
    REQUIREMENTS_PATH,
    requirements_text
)


# ------------------------------------------------------------
# Final metric rows
# ------------------------------------------------------------
image_metrics_row = (
    saved_image_metrics.iloc[0]
)

source_metrics_row = (
    saved_source_metrics.iloc[0]
)


# ------------------------------------------------------------
# Human-readable summary
# ------------------------------------------------------------
RUN_SUMMARY_PATH = (
    RUN_ROOT /
    "RUN_SUMMARY.txt"
)

run_summary_text = f"""
DENSENET121 MOUTH ROI DEEPFAKE EXPERIMENT
=========================================

Run ID
------
{RUN_ID}

Dataset
-------
Training images   : 2389
Validation images : 296
Test images       : 302
Total ROI images  : 2987

Model
-----
Architecture      : ImageNet-pretrained DenseNet121
Input             : 224 x 224 RGB mouth ROI
Frozen stage      : 12 epochs
Fine-tuning stage : 10 epochs
Global best epoch : {finetune_summary["global_best_epoch"]}
Best val ROC-AUC  : {finetune_summary["global_best_val_auc"]:.6f}

Threshold Selection
-------------------
Dataset           : Validation
Method            : Youden J
Threshold         : {SELECTED_THRESHOLD:.6f}
Test used         : No

Image-Level Independent Test Results
------------------------------------
Accuracy          : {float(image_metrics_row["accuracy"]):.6f}
Balanced Accuracy : {float(image_metrics_row["balanced_accuracy"]):.6f}
Precision         : {float(image_metrics_row["precision"]):.6f}
Recall            : {float(image_metrics_row["recall"]):.6f}
Specificity       : {float(image_metrics_row["specificity"]):.6f}
F1-score          : {float(image_metrics_row["f1_score"]):.6f}
ROC-AUC           : {float(image_metrics_row["roc_auc"]):.6f}
PR-AUC            : {float(image_metrics_row["pr_auc"]):.6f}

Source-Level Independent Test Results
-------------------------------------
Source count      : {int(source_metrics_row["sample_count"])}
Accuracy          : {float(source_metrics_row["accuracy"]):.6f}
Balanced Accuracy : {float(source_metrics_row["balanced_accuracy"]):.6f}
Precision         : {float(source_metrics_row["precision"]):.6f}
Recall            : {float(source_metrics_row["recall"]):.6f}
Specificity       : {float(source_metrics_row["specificity"]):.6f}
F1-score          : {float(source_metrics_row["f1_score"]):.6f}
ROC-AUC           : {float(source_metrics_row["roc_auc"]):.6f}
PR-AUC            : {float(source_metrics_row["pr_auc"]):.6f}

Quality Gates
-------------
Metadata accounting       : PASS
Image-metadata matching   : PASS
Source-video leakage      : PASS
SHA-256 leakage           : PASS
Two-batch training smoke  : PASS
Gradient NaN/Inf          : PASS
Atomic checkpoint         : PASS
Checkpoint reload         : PASS
Clean-load inference      : PASS
Figure resolution         : PASS

Label Mapping
-------------
Real = 0
Fake = 1
""".strip() + "\n"

atomic_write_text(
    RUN_SUMMARY_PATH,
    run_summary_text
)


# ------------------------------------------------------------
# Machine-readable final audit
# ------------------------------------------------------------
FINAL_AUDIT_PATH = (
    RUN_ROOT /
    "final_audit.json"
)

image_metrics_dictionary = {
    column_name: float(
        image_metrics_row[column_name]
    )
    for column_name in (
        saved_image_metrics.columns
    )
}

source_metrics_dictionary = {
    column_name: float(
        source_metrics_row[column_name]
    )
    for column_name in (
        saved_source_metrics.columns
    )
}

final_audit = {
    "run_id": RUN_ID,
    "status": "COMPLETED",
    "model": "DenseNet121",
    "best_model_path": str(
        BEST_MODEL_PATH
    ),
    "best_model_exists": True,
    "last_model_exists": True,
    "manifest_rows": int(
        len(saved_manifest)
    ),
    "test_prediction_rows": int(
        len(saved_image_predictions)
    ),
    "source_prediction_rows": int(
        len(saved_source_predictions)
    ),
    "combined_training_epochs": int(
        len(saved_combined_history)
    ),
    "source_video_leakage_pass": True,
    "sha256_leakage_pass": True,
    "clean_load_inference_pass": True,
    "figure_quality_pass": True,
    "test_used_only_for_final_evaluation": True,
    "image_level_metrics": (
        image_metrics_dictionary
    ),
    "source_level_metrics": (
        source_metrics_dictionary
    )
}

atomic_write_json(
    FINAL_AUDIT_PATH,
    final_audit
)


# ------------------------------------------------------------
# Artifact checksum manifest
# ------------------------------------------------------------
ARTIFACT_MANIFEST_PATH = (
    RUN_ROOT /
    "artifact_manifest.csv"
)

artifact_files = sorted(
    [
        file_path
        for file_path in RUN_ROOT.rglob("*")
        if (
            file_path.is_file()
            and file_path !=
            ARTIFACT_MANIFEST_PATH
        )
    ],
    key=lambda file_path: str(
        file_path.relative_to(
            RUN_ROOT
        )
    )
)

artifact_records = []

print("\nCalculating artifact checksums...")

for artifact_number, artifact_path in enumerate(
    artifact_files,
    start=1
):
    artifact_records.append(
        {
            "relative_path": str(
                artifact_path.relative_to(
                    RUN_ROOT
                )
            ),
            "size_bytes": int(
                artifact_path.stat().st_size
            ),
            "sha256": calculate_sha256(
                artifact_path
            )
        }
    )

    if (
        artifact_number % 10 == 0
        or artifact_number ==
        len(artifact_files)
    ):
        print(
            f"Artifacts hashed: "
            f"{artifact_number} / "
            f"{len(artifact_files)}"
        )

atomic_write_csv(
    pd.DataFrame(
        artifact_records
    ),
    ARTIFACT_MANIFEST_PATH
)


# ------------------------------------------------------------
# ZIP paths
# ------------------------------------------------------------
ZIP_PATH = (
    RUN_ROOT.parent /
    f"{RUN_ID}.zip"
)

TEMP_ZIP_BASE = (
    RUN_ROOT.parent /
    f"{RUN_ID}.tmp"
)

TEMP_ZIP_PATH = Path(
    str(TEMP_ZIP_BASE) +
    ".zip"
)

ZIP_SHA256_PATH = (
    RUN_ROOT.parent /
    f"{RUN_ID}.zip.sha256"
)

if TEMP_ZIP_PATH.exists():
    TEMP_ZIP_PATH.unlink()


# ------------------------------------------------------------
# Create ZIP
# ------------------------------------------------------------
print("\nCreating final ZIP archive...")

created_zip_path = shutil.make_archive(
    base_name=str(
        TEMP_ZIP_BASE
    ),
    format="zip",
    root_dir=str(
        RUN_ROOT.parent
    ),
    base_dir=RUN_ROOT.name
)

created_zip_path = Path(
    created_zip_path
)

assert created_zip_path == TEMP_ZIP_PATH

assert TEMP_ZIP_PATH.exists()


# ------------------------------------------------------------
# ZIP integrity verification
# ------------------------------------------------------------
with zipfile.ZipFile(
    TEMP_ZIP_PATH,
    mode="r"
) as zip_file:

    corrupt_member = zip_file.testzip()

    assert corrupt_member is None, (
        f"Corrupt ZIP member: "
        f"{corrupt_member}"
    )

    zip_member_count = len(
        zip_file.namelist()
    )

    assert zip_member_count > 0


# ------------------------------------------------------------
# Publish ZIP atomically
# ------------------------------------------------------------
os.replace(
    TEMP_ZIP_PATH,
    ZIP_PATH
)

assert ZIP_PATH.exists()
assert ZIP_PATH.stat().st_size > 0


# ------------------------------------------------------------
# ZIP checksum
# ------------------------------------------------------------
zip_sha256 = calculate_sha256(
    ZIP_PATH
)

atomic_write_text(
    ZIP_SHA256_PATH,
    (
        f"{zip_sha256}  "
        f"{ZIP_PATH.name}\n"
    )
)

zip_size_mb = (
    ZIP_PATH.stat().st_size /
    (1024 ** 2)
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("DENSENET121 EXPERIMENT — FINAL AUDIT COMPLETE")
print("=" * 100)

print("Run ID            :", RUN_ID)
print("Run folder        :", RUN_ROOT)
print("Best model        :", BEST_MODEL_PATH)
print("Final audit       :", FINAL_AUDIT_PATH)
print("Artifact manifest :", ARTIFACT_MANIFEST_PATH)
print("ZIP archive       :", ZIP_PATH)
print(
    "ZIP size          :",
    f"{zip_size_mb:.2f} MB"
)
print("ZIP members       :", zip_member_count)
print("ZIP SHA-256       :", zip_sha256)

print("\nFinal image-level metrics:")

print(
    saved_image_metrics.to_string(
        index=False
    )
)

print("=" * 100)
print("REQUIRED FILE AUDIT       : PASS")
print("TABLE QUALITY GATES       : PASS")
print("CLEAN-LOAD INFERENCE      : PASS")
print("FIGURE QUALITY GATES      : PASS")
print("ARTIFACT CHECKSUMS        : SAVED")
print("ZIP INTEGRITY TEST        : PASS")
print("EXPERIMENT STATUS         : COMPLETED")
print("CELL 15 COMPLETED")
print("=" * 100)


FINAL AUDIT, CLEAN-LOAD INFERENCE TEST AND ZIP EXPORT

Required directories:
----------------------------------------------------------------------------------------------------
checkpoints    : FOUND
logs           : FOUND
metrics        : FOUND
predictions    : FOUND
figures        : FOUND
metadata       : FOUND

Required files:
----------------------------------------------------------------------------------------------------
resolved_config             : FOUND
best_model                  : FOUND
last_model                  : FOUND
model_manifest              : FOUND
leakage_report              : FOUND
accounting_report           : FOUND
smoke_test_report           : FOUND
frozen_history              : FOUND
finetune_history            : FOUND
combined_history            : FOUND
image_metrics               : FOUND
source_metrics              : FOUND
classification_report       : FOUND
validation_threshold        : FOUND
final_evaluation            : FOUND
image_predictions        

In [4]:
# ============================================================
# FIGURE FILE CHECK
# ============================================================

from pathlib import Path

print("Figure folder:", FIGURE_DIR)
print("\nExisting figure files:")

figure_files = sorted(
    [
        file_path
        for file_path in FIGURE_DIR.iterdir()
        if file_path.is_file()
    ],
    key=lambda path: path.name.lower()
)

if not figure_files:
    print("No figure files found.")
else:
    for file_path in figure_files:
        print(
            file_path.name,
            "|",
            f"{file_path.stat().st_size / 1024:.2f} KB"
        )

Figure folder: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/figures

Existing figure files:
test_confusion_matrices_matrices.png | 42.51 KB
test_probability_distribution.png | 49.58 KB
test_roc_pr_curves.png | 96.01 KB
training_validation_curves.png | 258.90 KB


In [5]:
# ============================================================
# FIX CONFUSION MATRIX FILENAME
# ============================================================

import os
from pathlib import Path

old_confusion_path = (
    FIGURE_DIR /
    "test_confusion_matrices_matrices.png"
)

correct_confusion_path = (
    FIGURE_DIR /
    "test_confusion_matrices.png"
)

assert old_confusion_path.exists(), (
    f"Source figure was not found:\n"
    f"{old_confusion_path}"
)

assert old_confusion_path.stat().st_size > 0, (
    "Source confusion matrix figure is empty."
)

os.replace(
    old_confusion_path,
    correct_confusion_path
)

assert correct_confusion_path.exists()
assert correct_confusion_path.stat().st_size > 0

print(
    "Corrected figure:",
    correct_confusion_path
)

print("CONFUSION MATRIX FILENAME FIX: PASS")

Corrected figure: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/DenseNet121_Mouth_Results/20260808_0856_mouth_densenet121_seed42/figures/test_confusion_matrices.png
CONFUSION MATRIX FILENAME FIX: PASS
